In [267]:
import os
import sys
import pandas as pd
from urllib.parse import quote_plus
from sqlalchemy import create_engine, inspect

project_root = os.path.abspath("../..")  # ajuste se necessário
sys.path.append(project_root)

from dotenv import load_dotenv

load_dotenv()

True

In [268]:
def obter_engine():
    DB_USER = os.getenv("POSTGRES_USER")
    DB_PASSWORD_RAW = os.getenv("POSTGRES_PASSWORD")
    DB_HOST = os.getenv("POSTGRES_HOST")
    DB_PORT = os.getenv("POSTGRES_PORT")
    DB_NAME = os.getenv("POSTGRES_DB")
    REQUISITOS_SSL = os.getenv("REQUISITOS_SSL", "")

    if not all([DB_USER, DB_PASSWORD_RAW, DB_HOST, DB_PORT, DB_NAME]):
        raise EnvironmentError("Variáveis de ambiente incompletas no .env")

    DB_PASSWORD = quote_plus(DB_PASSWORD_RAW)

    DATABASE_URL = (
        f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
        f"@{DB_HOST}:{DB_PORT}/{DB_NAME}{REQUISITOS_SSL}"
    )

    engine = create_engine(DATABASE_URL, pool_pre_ping=True)
    return engine


# 🔹 Função genérica para carregar qualquer tabela
def carregar_tabela(nome_tabela, limite=None):
    engine = obter_engine()

    query = f"SELECT * FROM {nome_tabela}"
    if limite:
        query += f" LIMIT {limite}"

    df = pd.read_sql(query, engine)
    engine.dispose()
    return df


# 🔹 Lista todas as tabelas do banco
def listar_tabelas():
    engine = obter_engine()
    inspector = inspect(engine)
    tabelas = inspector.get_table_names()
    engine.dispose()
    return tabelas

def analisar_tabela(nome_tabela):
    print("=" * 60)
    print(f"📂 TABELA: {nome_tabela}")
    print("=" * 60)

    df = carregar_tabela(nome_tabela)

    print(f"🔢 Linhas: {df.shape[0]}")
    print(f"📊 Colunas: {df.shape[1]}")
    print("\n🧩 Tipos de dados:")
    print(df.dtypes)

    print("\n❓ Valores nulos:")
    print(df.isnull().sum())

    print("\n🔎 Estatísticas numéricas")
    display(df.describe())

    print("\n👀 Primeiras linhas:")
    display(df.head())

    print("\n\n")

# Análise Automática das Tabelas Silver
def resumo_tabelas():
    resultados = []

    for tabela in listar_tabelas():
        df = carregar_tabela(tabela)

        resultados.append(
            {
                "tabela": tabela,
                "linhas": df.shape[0],
                "colunas": df.shape[1],
                "valores_nulos_total": df.isnull().sum().sum(),
            }
        )

    return pd.DataFrame(resultados)

def renomear_linha(dataset, coluna, registro_principal, registro_nomeado):
    dataset.loc[
        dataset[coluna] == registro_principal,
        coluna,
    ] = registro_nomeado


def recriar_regiao_com_valor(
    df: pd.DataFrame,
    nome_regiao: str,
    coluna_regiao: str = "região_administrativa",
    coluna_ano: str = "ano",
    coluna_valor: str = "crimes_contra_mulher",
    valor_padrao: int | float = 0,
) -> pd.DataFrame:
    """
    Remove registros de uma região específica e recria
    a região para todos os anos existentes no dataset,
    preenchendo com um valor padrão.
    """

    # Captura todos os anos únicos
    anos = df[coluna_ano].unique()

    # Cria novo dataframe base para a região
    df_base = pd.DataFrame(
        {coluna_ano: anos, coluna_regiao: nome_regiao, coluna_valor: valor_padrao}
    )

    # Remove registros antigos da região
    df_filtrado = df[df[coluna_regiao] != nome_regiao]

    # Junta novamente
    df_final = pd.concat([df_base, df_filtrado], ignore_index=True)

    return df_final

In [269]:
# Listar suas 21 tabelas
tabelas = listar_tabelas()
tabelas


['crimes_contra_mulher',
 'desaparecidos_idade_sexo',
 'desaparecimento_localizados',
 'desaparecimento_regiao',
 'populacao_df',
 'feminicidio',
 'furto_em_veiculo',
 'homicidio',
 'violencia_idosos',
 'violencia_idosos_mensais',
 'violencia_idosos_ocorrencias',
 'violencia_idosos_sexo',
 'injuria_racial',
 'latrocinio',
 'lesao_corporal_morte',
 'populacao_regiao_administrativa',
 'racismo',
 'roubo_comercio',
 'roubo_transporte_coletivo',
 'roubo_veiculo',
 'roubo_pedestre']

In [270]:
# 🔥 Executa análise em todas as tabelas
for tabela in listar_tabelas():
    analisar_tabela(tabela)


📂 TABELA: crimes_contra_mulher
🔢 Linhas: 230
📊 Colunas: 9

🧩 Tipos de dados:
data_do_crime                  object
ra                             object
#_casos                         int64
meio_utilizado                 object
local                          object
motivação                      object
idade___vítima                 object
idade___autor                  object
inserido_em       datetime64[ns, UTC]
dtype: object

❓ Valores nulos:
data_do_crime     0
ra                0
#_casos           0
meio_utilizado    0
local             0
motivação         0
idade___vítima    0
idade___autor     0
inserido_em       0
dtype: int64

🔎 Estatísticas numéricas


,#_casos
count,230.0
mean,1.0
std,0.0
min,1.0
25%,1.0
50%,1.0
75%,1.0
max,1.0



👀 Primeiras linhas:


,data_do_crime,ra,#_casos,meio_utilizado,local,motivação,idade___vítima,idade___autor,inserido_em
0,2020-07-30,ÁGUAS CLARAS,1,Arma Branca,Interior de Residência,Não Informado,34,41,2026-02-14 03:03:33.592301+00:00
1,2024-05-31,ÁGUAS CLARAS,1,Asfixia,Interior de Residência,Outro,94,64,2026-02-14 03:03:33.592301+00:00
2,2023-08-11,ARNIQUEIRAS,1,Arma Branca,Interior de Residência,Término Relacionamento,45,46,2026-02-14 03:03:33.592301+00:00
3,2022-01-25,BRAZLÂNDIA,1,Agressão Física,Interior de Residência,Término Relacionamento,23,46,2026-02-14 03:03:33.592301+00:00
4,2015-05-18,BRAZLÂNDIA,1,Arma Branca,Interior de Residência,Ciúme,52,54,2026-02-14 03:03:33.592301+00:00





📂 TABELA: desaparecidos_idade_sexo
🔢 Linhas: 60
📊 Colunas: 5

🧩 Tipos de dados:
ano                           int64
faixa_etaria                 object
sexo                         object
quantidade                    int64
inserido_em     datetime64[ns, UTC]
dtype: object

❓ Valores nulos:
ano             0
faixa_etaria    0
sexo            0
quantidade      0
inserido_em     0
dtype: int64

🔎 Estatísticas numéricas


...

,Região Administrativa,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,inserido_em
0,Distrito Federal,4808.0,5663.0,4851.0,3993.0,3424.0,2218.0,2033.0,1553.0,1291.0,1018.0,2026-02-14 03:03:34.562131+00:00
1,Águas Claras,105.0,103.0,83.0,48.0,24.0,14.0,19.0,9.0,13.0,18.0,2026-02-14 03:03:34.562131+00:00
2,Arniqueira,0.0,0.0,0.0,0.0,0.0,30.0,19.0,20.0,12.0,6.0,2026-02-14 03:03:34.562131+00:00
3,Brasília,303.0,294.0,351.0,262.0,134.0,78.0,56.0,45.0,45.0,32.0,2026-02-14 03:03:34.562131+00:00
4,Brazlândia,85.0,71.0,81.0,74.0,55.0,39.0,34.0,26.0,24.0,18.0,2026-02-14 03:03:34.562131+00:00





📂 TABELA: roubo_pedestre
🔢 Linhas: 35
📊 Colunas: 12

🧩 Tipos de dados:
Região Administrativa                 object
2015                                 float64
2016                                 float64
2017                                 float64
2018                                 float64
2019                                 float64
2020                                 float64
2021                                 float64
2022                                 float64
2023                                 float64
2024                                 float64
inserido_em              datetime64[ns, UTC]
dtype: object

❓ Valores nulos:
Região Administrativa    0
2015                     0
2016                     0
2017                     0
2018                     0
2019                     0
2020                     0
2021                     0
2022                     0
2023                     0
2024                     0
inserido_em              0
dtype: int64

🔎 Estatísticas n

,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
count,35.000000,35.000000,35.000000,35.000000,35.000000,35.000000,35.000000,35.000000,35.000000,35.000000
mean,1730.285714,2183.200000,2100.285714,1886.000000,1672.857143,1126.685714,957.942857,951.657143,730.342857,609.085714
std,5131.635409,6450.561365,6207.638703,5582.789989,4960.820563,3327.618560,2840.413339,2828.133431,2161.450955,1804.453272
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,75.000000,82.500000,75.500000,69.000000,71.500000,59.500000,48.500000,49.500000,33.000000,24.500000
50%,322.000000,460.000000,467.000000,415.000000,273.000000,281.000000,201.000000,170.000000,174.000000,164.000000
75%,1354.500000,1700.000000,1617.500000,1648.500000,1316.000000,799.500000,662.000000,728.000000,590.000000,457.500000
max,30280.000000,38206.000000,36755.000000,33005.000000,29275.000000,19717.000000,16764.000000,16654.000000,12781.000000,10659.000000



👀 Primeiras linhas:


,Região Administrativa,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,inserido_em
0,Distrito Federal,30280.0,38206.0,36755.0,33005.0,29275.0,19717.0,16764.0,16654.0,12781.0,10659.0,2026-02-14 03:03:34.607194+00:00
1,Águas Claras,495.0,544.0,456.0,427.0,197.0,164.0,151.0,101.0,119.0,187.0,2026-02-14 03:03:34.607194+00:00
2,Arniqueira,0.0,0.0,0.0,0.0,0.0,185.0,114.0,120.0,84.0,44.0,2026-02-14 03:03:34.607194+00:00
3,Brasília,2417.0,3490.0,3119.0,3169.0,2975.0,1674.0,1441.0,1722.0,1454.0,991.0,2026-02-14 03:03:34.607194+00:00
4,Brazlândia,369.0,460.0,489.0,332.0,271.0,200.0,136.0,155.0,113.0,61.0,2026-02-14 03:03:34.607194+00:00


,tabela,linhas,colunas,valores_nulos_total
0,crimes_contra_mulher,230,9,0
1,desaparecidos_idade_sexo,60,5,0
2,desaparecimento_localizados,12,5,0
3,desaparecimento_regiao,33,6,0
4,populacao_df,18,6,1
5,feminicidio,34,12,0
6,furto_em_veiculo,33,12,0
7,homicidio,35,12,18
8,violencia_idosos,31,7,0
9,violencia_idosos_mensais,24,7,0


In [272]:
df_crime_contra_mulher = carregar_tabela("crimes_contra_mulher")
df_feminicidio = carregar_tabela("feminicidio")

In [273]:
display(df_feminicidio.head(230))


,região_administrativa,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,inserido_em
0,Distrito Federal,9,22,17,27,33,18,26,20,31,23,2026-02-14 03:03:33.957714+00:00
1,Águas Claras,0,0,1,0,0,1,0,0,0,1,2026-02-14 03:03:33.957714+00:00
2,Arniqueira,-,-,-,-,-,0,0,0,1,0,2026-02-14 03:03:33.957714+00:00
3,Brasília (Plano Piloto),0,1,1,2,4,0,0,0,1,0,2026-02-14 03:03:33.957714+00:00
4,Brazlândia,1,1,0,0,0,0,0,1,0,0,2026-02-14 03:03:33.957714+00:00
5,Candangolândia,0,0,1,0,0,1,0,0,0,0,2026-02-14 03:03:33.957714+00:00
6,Ceilândia,1,5,2,0,2,3,3,3,6,3,2026-02-14 03:03:33.957714+00:00
7,Cruzeiro,0,0,0,0,1,0,0,1,0,0,2026-02-14 03:03:33.957714+00:00
8,Fercal,0,1,1,0,1,1,0,0,0,0,2026-02-14 03:03:33.957714+00:00
9,Gama,2,2,0,1,2,0,1,0,3,3,2026-02-14 03:03:33.957714+00:00


In [274]:
df_femin_ = df_feminicidio.melt(
    id_vars="região_administrativa",  # coluna fixa
    var_name="ano",  # nome da nova coluna de anos
    value_name="casos_feminicidio",  # nome da coluna de valores
)


In [275]:
df_femin_.columns

Index(['região_administrativa', 'ano', 'casos_feminicidio'], dtype='object')

In [276]:
df_femin_[["ano", "casos_feminicidio"]] = (
    df_femin_[["ano", "casos_feminicidio"]]
    .apply(pd.to_numeric, errors="coerce")
    .astype("Int64")
)


In [277]:
df_femin_.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 374 entries, 0 to 373
Data columns (total 3 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   região_administrativa  374 non-null    object
 1   ano                    340 non-null    Int64 
 2   casos_feminicidio      330 non-null    Int64 
dtypes: Int64(2), object(1)
memory usage: 9.6+ KB


In [278]:
df_femin_["casos_feminicidio"] = df_femin_["casos_feminicidio"].fillna(0)


In [279]:
df_femin_.groupby("ano")["casos_feminicidio"].sum()


ano
2015    18
2016    44
2017    34
2018    54
2019    66
2020    36
2021    52
2022    40
2023    62
2024    46
Name: casos_feminicidio, dtype: Int64

In [280]:
df_femin_["região_administrativa"] = df_femin_["região_administrativa"].str.upper()

In [281]:
df_femin_.groupby("região_administrativa")["casos_feminicidio"].sum()


região_administrativa
ARNIQUEIRA                   1
BRASÍLIA (PLANO PILOTO)      9
BRAZLÂNDIA                   3
CANDANGOLÂNDIA               2
CEILÂNDIA                   28
CRUZEIRO                     2
DISTRITO FEDERAL           226
FERCAL                       4
GAMA                        14
GUARÁ                        4
ITAPOÃ                      10
JARDIM BOTÂNICO              2
LAGO NORTE                   0
LAGO SUL                     1
NÚCLEO BANDEIRANTE           1
PARANOÁ                      8
PARK WAY                     1
PLANALTINA                  15
RECANTO DAS EMAS            11
RIACHO FUNDO                 3
RIACHO FUNDO II              6
SAMAMBAIA                   21
SANTA MARIA                 15
SCIA/ESTRUTURAL              8
SIA                          1
SOBRADINHO                  12
SOBRADINHO II                7
SOL NASCENTE/PÔR DO SOL      5
SUDOESTE/OCTOGONAL           2
SÃO SEBASTIÃO                9
TAGUATINGA                  11
VARJÃO           

In [282]:
df_femin_['região_administrativa'] == "DISTRITO FEDERAL"


0       True
1      False
2      False
3      False
4      False
       ...  
369    False
370    False
371    False
372    False
373    False
Name: região_administrativa, Length: 374, dtype: bool

In [283]:
df_femin_ = df_femin_[df_femin_["região_administrativa"] != "DISTRITO FEDERAL"]


In [284]:
df_femin_["região_administrativa"].unique()

array(['ÁGUAS CLARAS', 'ARNIQUEIRA', 'BRASÍLIA (PLANO PILOTO)',
       'BRAZLÂNDIA', 'CANDANGOLÂNDIA', 'CEILÂNDIA', 'CRUZEIRO', 'FERCAL',
       'GAMA', 'GUARÁ', 'ITAPOÃ', 'JARDIM BOTÂNICO', 'LAGO NORTE',
       'LAGO SUL', 'NÚCLEO BANDEIRANTE', 'PARANOÁ', 'PARK WAY',
       'PLANALTINA', 'RECANTO DAS EMAS', 'RIACHO FUNDO',
       'RIACHO FUNDO II', 'SAMAMBAIA', 'SANTA MARIA', 'SÃO SEBASTIÃO',
       'SCIA/ESTRUTURAL', 'SIA', 'SOBRADINHO', 'SOBRADINHO II',
       'SOL NASCENTE/PÔR DO SOL', 'SUDOESTE/OCTOGONAL', 'TAGUATINGA',
       'VARJÃO', 'VICENTE PIRES'], dtype=object)

In [285]:
renomear_linha(
    df_femin_, "região_administrativa", "BRASÍLIA (PLANO PILOTO)", "PLANO PILOTO"
)

In [286]:
renomear_linha(
    df_femin_,
    "região_administrativa",
    "SIA",
    "SIA (SETOR DE INDUSTRIA E ABASTECIMENTO)",
)


In [287]:
renomear_linha(
    df_femin_,
    "região_administrativa",
    "SOL NASCENTE/PÔR DO SOL",
    "POR DO SOL/SOL NASCENTE",
)


In [ ]:
renomear_linha(
    df_femin_,
    "região_administrativa",
    "ARNIQUEIRA",
    "ARNIQUEIRAS",
)


In [289]:
len(list(df_femin_["região_administrativa"].unique()))


33

In [290]:
df_femin_["região_administrativa"] = (
    df_femin_["região_administrativa"].str.strip().str.upper()
)

df_femin_= df_femin_.sort_values(
    by="região_administrativa"
).reset_index(drop=True)


In [291]:
df_femin_

,região_administrativa,ano,casos_feminicidio
0,ARNIQUEIRAS,2018,0
1,ARNIQUEIRAS,2016,0
2,ARNIQUEIRAS,2023,1
3,ARNIQUEIRAS,2022,0
4,ARNIQUEIRAS,2024,0
...,...,...,...
358,ÁGUAS CLARAS,2023,0
359,ÁGUAS CLARAS,2017,1
360,ÁGUAS CLARAS,2016,0
361,ÁGUAS CLARAS,2024,1


In [292]:
df_femin_ = df_femin_[["ano", "região_administrativa", "casos_feminicidio"]]


In [293]:
display(df_crime_contra_mulher.head())


,data_do_crime,ra,#_casos,meio_utilizado,local,motivação,idade___vítima,idade___autor,inserido_em
0,2020-07-30,ÁGUAS CLARAS,1,Arma Branca,Interior de Residência,Não Informado,34,41,2026-02-14 03:03:33.592301+00:00
1,2024-05-31,ÁGUAS CLARAS,1,Asfixia,Interior de Residência,Outro,94,64,2026-02-14 03:03:33.592301+00:00
2,2023-08-11,ARNIQUEIRAS,1,Arma Branca,Interior de Residência,Término Relacionamento,45,46,2026-02-14 03:03:33.592301+00:00
3,2022-01-25,BRAZLÂNDIA,1,Agressão Física,Interior de Residência,Término Relacionamento,23,46,2026-02-14 03:03:33.592301+00:00
4,2015-05-18,BRAZLÂNDIA,1,Arma Branca,Interior de Residência,Ciúme,52,54,2026-02-14 03:03:33.592301+00:00


In [294]:
df_crime_contra_mulher["data_do_crime"] = pd.to_datetime(
    df_crime_contra_mulher["data_do_crime"]
)


In [295]:
df_crime_contra_mulher["ano"] = df_crime_contra_mulher["data_do_crime"].dt.year
df_crime_contra_mulher

,data_do_crime,ra,#_casos,meio_utilizado,local,motivação,idade___vítima,idade___autor,inserido_em,ano
0,2020-07-30,ÁGUAS CLARAS,1,Arma Branca,Interior de Residência,Não Informado,34,41,2026-02-14 03:03:33.592301+00:00,2020
1,2024-05-31,ÁGUAS CLARAS,1,Asfixia,Interior de Residência,Outro,94,64,2026-02-14 03:03:33.592301+00:00,2024
2,2023-08-11,ARNIQUEIRAS,1,Arma Branca,Interior de Residência,Término Relacionamento,45,46,2026-02-14 03:03:33.592301+00:00,2023
3,2022-01-25,BRAZLÂNDIA,1,Agressão Física,Interior de Residência,Término Relacionamento,23,46,2026-02-14 03:03:33.592301+00:00,2022
4,2015-05-18,BRAZLÂNDIA,1,Arma Branca,Interior de Residência,Ciúme,52,54,2026-02-14 03:03:33.592301+00:00,2015
...,...,...,...,...,...,...,...,...,...,...
225,2024-11-11,VICENTE PIRES,1,Arma Branca,Interior de Residência,Término Relacionamento,30,29,2026-02-14 03:03:33.592301+00:00,2024
226,2024-09-30,VICENTE PIRES,1,Arma de Fogo,Interior de Residência,Ciúme,26,32,2026-02-14 03:03:33.592301+00:00,2024
227,2017-03-16,VICENTE PIRES,1,Arma de Fogo,Interior de Residência,Término Relacionamento,29,39,2026-02-14 03:03:33.592301+00:00,2017
228,2019-10-18,VICENTE PIRES,1,Arma de Fogo,"Lote vago, terreno abandonado",Ciúme,38,43,2026-02-14 03:03:33.592301+00:00,2019


In [296]:
df_crime_contra_mulher.groupby("ano")["#_casos"].sum()


ano
2015    11
2016    21
2017    12
2018    25
2019    29
2020    17
2021    23
2022    17
2023    30
2024    22
2025    23
Name: #_casos, dtype: int64

In [297]:
display(df_crime_contra_mulher.groupby( "ra")["#_casos"].sum())



ra
ARNIQUEIRAS                                  1
BRAZLÂNDIA                                   4
CANDANGOLÂNDIA                               2
CEILÂNDIA                                   30
CRUZEIRO                                     2
FERCAL                                       3
GAMA                                        12
GUARÁ                                        3
ITAPOÃ                                      10
JARDIM BOTANICO                              2
LAGO SUL                                     1
NÚCLEO BANDEIRANTE                           3
PARANOÁ                                     10
PARK WAY                                     2
PLANALTINA                                  17
PLANO PILOTO                                 9
POR DO SOL/SOL NASCENTE                      9
RECANTO DAS EMAS                            11
RIACHO FUNDO                                 2
RIACHO FUNDO II                              6
SAMAMBAIA                                   21
SANTA MARI

In [298]:
display(df_crime_contra_mulher["#_casos"].sum())


np.int64(230)

In [299]:
df_crime_contra_mulher_ = df_crime_contra_mulher.groupby(["ano", "ra"])["#_casos"].sum().reset_index()


In [300]:
df_crime_contra_mulher_.rename(columns={"ra": "região_administrativa"}, inplace=True)


In [301]:
df_crime_contra_mulher_.rename(
    columns={"#_casos": "crimes_contra_mulher"}, inplace=True
)


In [302]:
renomear_linha(
    df_crime_contra_mulher_,
    "região_administrativa",
    "JARDIM BOTANICO",
    "JARDIM BOTÂNICO",
)

In [303]:
renomear_linha(
    df_crime_contra_mulher_,
    "região_administrativa",
    "SCIA E ESTRUTURAL",
    "SCIA/ESTRUTURAL",
)


In [304]:
renomear_linha(
    df_crime_contra_mulher_,
    "região_administrativa",
    "SUDOESTE",
    "SUDOESTE/OCTOGONAL",
)


In [305]:
renomear_linha(
    df_crime_contra_mulher_,
    "região_administrativa",
    "SUDOESTE",
    "SUDOESTE/OCTOGONAL",
)


In [306]:
df_crime_contra_mulher_.loc[
    df_crime_contra_mulher_["região_administrativa"] == "LAGO MORTE",
    "região_administrativa",
] = "LAGO NORTE"


In [307]:
df_crime_contra_mulher_["região_administrativa"].unique()


array(['BRAZLÂNDIA', 'CEILÂNDIA', 'GAMA', 'GUARÁ', 'PLANALTINA',
       'RIACHO FUNDO II', 'SAMAMBAIA', 'SCIA/ESTRUTURAL', 'SOBRADINHO II',
       'FERCAL', 'ITAPOÃ', 'JARDIM BOTÂNICO', 'PLANO PILOTO',
       'POR DO SOL/SOL NASCENTE', 'SANTA MARIA', 'SOBRADINHO',
       'CANDANGOLÂNDIA', 'RIACHO FUNDO', 'SÃO SEBASTIÃO', 'VICENTE PIRES',
       'RECANTO DAS EMAS', 'CRUZEIRO', 'PARANOÁ', 'SUDOESTE/OCTOGONAL',
       'TAGUATINGA', 'LAGO SUL', 'NÚCLEO BANDEIRANTE', 'ÁGUAS CLARAS',
       'ARNIQUEIRAS', 'PARK WAY',
       'SIA (SETOR DE INDUSTRIA E ABASTECIMENTO)'], dtype=object)

In [308]:
len(list(df_crime_contra_mulher_["região_administrativa"].unique()))


31

In [309]:
df_crime_contra_mulher_ = recriar_regiao_com_valor(
    df=df_crime_contra_mulher_, nome_regiao="VARJÃO", valor_padrao=0
)


In [310]:
df_crime_contra_mulher_ = recriar_regiao_com_valor(
    df=df_crime_contra_mulher_, nome_regiao="LAGO NORTE", valor_padrao=0
)


In [311]:
df_crime_contra_mulher_['região_administrativa'].unique()


array(['LAGO NORTE', 'VARJÃO', 'BRAZLÂNDIA', 'CEILÂNDIA', 'GAMA', 'GUARÁ',
       'PLANALTINA', 'RIACHO FUNDO II', 'SAMAMBAIA', 'SCIA/ESTRUTURAL',
       'SOBRADINHO II', 'FERCAL', 'ITAPOÃ', 'JARDIM BOTÂNICO',
       'PLANO PILOTO', 'POR DO SOL/SOL NASCENTE', 'SANTA MARIA',
       'SOBRADINHO', 'CANDANGOLÂNDIA', 'RIACHO FUNDO', 'SÃO SEBASTIÃO',
       'VICENTE PIRES', 'RECANTO DAS EMAS', 'CRUZEIRO', 'PARANOÁ',
       'SUDOESTE/OCTOGONAL', 'TAGUATINGA', 'LAGO SUL',
       'NÚCLEO BANDEIRANTE', 'ÁGUAS CLARAS', 'ARNIQUEIRAS', 'PARK WAY',
       'SIA (SETOR DE INDUSTRIA E ABASTECIMENTO)'], dtype=object)

In [312]:
df_crime_contra_mulher_["região_administrativa"] = (
    df_crime_contra_mulher_["região_administrativa"].str.strip().str.upper()
)

df_crime_contra_mulher_ = df_crime_contra_mulher_.sort_values(
    by="região_administrativa"
).reset_index(drop=True)


In [313]:
df_crime_contra_mulher_

,ano,região_administrativa,crimes_contra_mulher
0,2023,ARNIQUEIRAS,1
1,2015,BRAZLÂNDIA,1
2,2016,BRAZLÂNDIA,1
3,2022,BRAZLÂNDIA,1
4,2025,BRAZLÂNDIA,1
...,...,...,...
163,2024,VICENTE PIRES,2
164,2020,VICENTE PIRES,1
165,2019,VICENTE PIRES,2
166,2024,ÁGUAS CLARAS,1


In [314]:
print("Comparação dos Crimes:")

print("\nCrime contra mulher\n")
display(df_crime_contra_mulher_.head())
display(df_crime_contra_mulher_.shape)

print("\nFeminicídio\n")
display(df_femin_.head())
display(df_femin_.shape)

Comparação dos Crimes:

Crime contra mulher



,ano,região_administrativa,crimes_contra_mulher
0,2023,ARNIQUEIRAS,1
1,2015,BRAZLÂNDIA,1
2,2016,BRAZLÂNDIA,1
3,2022,BRAZLÂNDIA,1
4,2025,BRAZLÂNDIA,1


(168, 3)


Feminicídio



,ano,região_administrativa,casos_feminicidio
0,2018,ARNIQUEIRAS,0
1,2016,ARNIQUEIRAS,0
2,2023,ARNIQUEIRAS,1
3,2022,ARNIQUEIRAS,0
4,2024,ARNIQUEIRAS,0


(363, 3)

In [315]:
df_violencia_mulher = pd.merge(
    df_crime_contra_mulher_, df_femin_, on=["ano", "região_administrativa"], how="outer"
)


In [316]:
df_violencia_mulher


,ano,região_administrativa,crimes_contra_mulher,casos_feminicidio
0,2015,ARNIQUEIRAS,NaN,0
1,2015,BRAZLÂNDIA,1.0,1
2,2015,CANDANGOLÂNDIA,NaN,0
3,2015,CEILÂNDIA,3.0,1
4,2015,CRUZEIRO,NaN,0
...,...,...,...,...
375,<NA>,SÃO SEBASTIÃO,NaN,0
376,<NA>,TAGUATINGA,NaN,0
377,<NA>,VARJÃO,NaN,0
378,<NA>,VICENTE PIRES,NaN,0


In [317]:
df_femin_['região_administrativa'].equals(df_crime_contra_mulher_['região_administrativa'])


False

In [318]:
set(df_femin_['região_administrativa']) - set(df_crime_contra_mulher_['região_administrativa'])


set()

In [319]:
colunas_femin = set(df_femin_['região_administrativa'])
colunas_crime = set(df_crime_contra_mulher_["região_administrativa"])

print("Só no df_femin_:")
print(colunas_femin - colunas_crime)

print("\nSó no df_crime_contra_mulher_:")
print(colunas_crime - colunas_femin)


Só no df_femin_:
set()

Só no df_crime_contra_mulher_:
set()


In [320]:
df_femin_['casos_feminicidio'].sum()

np.int64(226)

In [321]:
df_crime_contra_mulher_['crimes_contra_mulher'].sum()

np.int64(230)

In [322]:
df_violencia_mulher['casos_feminicidio'].sum()

np.int64(226)

In [323]:
df_violencia_mulher['crimes_contra_mulher'].sum()

np.float64(230.0)

In [324]:
df_violencia_mulher = df_violencia_mulher.fillna(0)


In [325]:
df_violencia_mulher

,ano,região_administrativa,crimes_contra_mulher,casos_feminicidio
0,2015,ARNIQUEIRAS,0.0,0
1,2015,BRAZLÂNDIA,1.0,1
2,2015,CANDANGOLÂNDIA,0.0,0
3,2015,CEILÂNDIA,3.0,1
4,2015,CRUZEIRO,0.0,0
...,...,...,...,...
375,0,SÃO SEBASTIÃO,0.0,0
376,0,TAGUATINGA,0.0,0
377,0,VARJÃO,0.0,0
378,0,VICENTE PIRES,0.0,0


In [326]:
df_violencia_mulher=df_violencia_mulher[df_violencia_mulher["ano"] != 0]


In [327]:
df_violencia_mulher["crimes_contra_mulher"] = df_violencia_mulher[
    "crimes_contra_mulher"
].astype(int)


C:\Users\Alexandre\AppData\Local\Temp\ipykernel_21284\2406250406.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_violencia_mulher["crimes_contra_mulher"] = df_violencia_mulher[


In [328]:
df_violencia_mulher


,ano,região_administrativa,crimes_contra_mulher,casos_feminicidio
0,2015,ARNIQUEIRAS,0,0
1,2015,BRAZLÂNDIA,1,1
2,2015,CANDANGOLÂNDIA,0,0
3,2015,CEILÂNDIA,3,1
4,2015,CRUZEIRO,0,0
...,...,...,...,...
342,2025,SAMAMBAIA,2,0
343,2025,SCIA/ESTRUTURAL,2,0
344,2025,SÃO SEBASTIÃO,1,0
345,2025,TAGUATINGA,1,0


In [329]:
df_violencia_mulher.groupby('ano')[['crimes_contra_mulher','casos_feminicidio']].sum()

,crimes_contra_mulher,casos_feminicidio
ano,,
2015,11,9
2016,21,22
2017,12,17
2018,25,27
2019,29,33
2020,17,18
2021,23,26
2022,17,20
2023,30,31


In [330]:
df_violencia_mulher.groupby("região_administrativa")[["crimes_contra_mulher", "casos_feminicidio"]].sum()


,crimes_contra_mulher,casos_feminicidio
região_administrativa,,
ARNIQUEIRAS,1,1
BRAZLÂNDIA,4,3
CANDANGOLÂNDIA,2,2
CEILÂNDIA,30,28
CRUZEIRO,2,2
FERCAL,3,4
GAMA,12,14
GUARÁ,3,4
ITAPOÃ,10,10


In [331]:
df_violencia_mulher.info()


<class 'pandas.core.frame.DataFrame'>
Index: 347 entries, 0 to 346
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   ano                    347 non-null    Int64 
 1   região_administrativa  347 non-null    object
 2   crimes_contra_mulher   347 non-null    int64 
 3   casos_feminicidio      347 non-null    Int64 
dtypes: Int64(2), int64(1), object(1)
memory usage: 14.2+ KB


In [332]:
df_violencia_mulher.describe()

,ano,crimes_contra_mulher,casos_feminicidio
count,347.0,347.000000,347.0
mean,2019.769452,0.662824,0.651297
std,3.04663,0.978896,1.021173
min,2015.0,0.000000,0.0
25%,2017.0,0.000000,0.0
50%,2020.0,0.000000,0.0
75%,2022.0,1.000000,1.0
max,2025.0,6.000000,6.0


In [333]:
df_violencia_mulher.describe(include='O')

,região_administrativa
count,347
unique,33
top,BRAZLÂNDIA
freq,11


In [334]:
df_crime_contra_mulher

,data_do_crime,ra,#_casos,meio_utilizado,local,motivação,idade___vítima,idade___autor,inserido_em,ano
0,2020-07-30,ÁGUAS CLARAS,1,Arma Branca,Interior de Residência,Não Informado,34,41,2026-02-14 03:03:33.592301+00:00,2020
1,2024-05-31,ÁGUAS CLARAS,1,Asfixia,Interior de Residência,Outro,94,64,2026-02-14 03:03:33.592301+00:00,2024
2,2023-08-11,ARNIQUEIRAS,1,Arma Branca,Interior de Residência,Término Relacionamento,45,46,2026-02-14 03:03:33.592301+00:00,2023
3,2022-01-25,BRAZLÂNDIA,1,Agressão Física,Interior de Residência,Término Relacionamento,23,46,2026-02-14 03:03:33.592301+00:00,2022
4,2015-05-18,BRAZLÂNDIA,1,Arma Branca,Interior de Residência,Ciúme,52,54,2026-02-14 03:03:33.592301+00:00,2015
...,...,...,...,...,...,...,...,...,...,...
225,2024-11-11,VICENTE PIRES,1,Arma Branca,Interior de Residência,Término Relacionamento,30,29,2026-02-14 03:03:33.592301+00:00,2024
226,2024-09-30,VICENTE PIRES,1,Arma de Fogo,Interior de Residência,Ciúme,26,32,2026-02-14 03:03:33.592301+00:00,2024
227,2017-03-16,VICENTE PIRES,1,Arma de Fogo,Interior de Residência,Término Relacionamento,29,39,2026-02-14 03:03:33.592301+00:00,2017
228,2019-10-18,VICENTE PIRES,1,Arma de Fogo,"Lote vago, terreno abandonado",Ciúme,38,43,2026-02-14 03:03:33.592301+00:00,2019


In [335]:
list(df_crime_contra_mulher.columns)

['data_do_crime',
 'ra',
 '#_casos',
 'meio_utilizado',
 'local',
 'motivação',
 'idade___vítima',
 'idade___autor',
 'inserido_em',
 'ano']

In [ ]:
identificacao_crimes_mulher = df_crime_contra_mulher[
    [
        "ano",
        "ra",
        "meio_utilizado",
        "local",
        "motivação",
        "idade___vítima",
        "idade___autor",
        "data_do_crime",
    ]
]

In [337]:
list(identificacao_crimes_mulher["ra"].unique())


['ÁGUAS CLARAS',
 'ARNIQUEIRAS',
 'BRAZLÂNDIA',
 'CANDANGOLÂNDIA',
 'CEILÂNDIA',
 'CRUZEIRO',
 'FERCAL',
 'GAMA',
 'GUARÁ',
 'ITAPOÃ',
 'JARDIM BOTANICO',
 'LAGO SUL',
 'NÚCLEO BANDEIRANTE',
 'PARANOÁ',
 'PARK WAY',
 'PLANALTINA',
 'PLANO PILOTO',
 'POR DO SOL/SOL NASCENTE',
 'RECANTO DAS EMAS',
 'RIACHO FUNDO',
 'RIACHO FUNDO II',
 'SAMAMBAIA',
 'SANTA MARIA',
 'SÃO SEBASTIÃO',
 'SCIA E ESTRUTURAL',
 'SIA (SETOR DE INDUSTRIA E ABASTECIMENTO)',
 'SOBRADINHO',
 'SOBRADINHO II',
 'SUDOESTE',
 'TAGUATINGA',
 'VICENTE PIRES']

In [338]:
renomear_linha(identificacao_crimes_mulher, 'ra', 'SUDOESTE', 'SUDOESTE/OCTOGONAL')

In [339]:
renomear_linha(identificacao_crimes_mulher, "ra", "JARDIM BOTANICO", "JARDIM BOTÂNICO")


In [340]:
renomear_linha(
    identificacao_crimes_mulher, "ra", "SCIA E ESTRUTURAL", "SCIA/ESTRUTURAL"
)


In [341]:
identificacao_crimes_mulher = recriar_regiao_com_valor(
    df=identificacao_crimes_mulher,coluna_regiao='ra' , nome_regiao="LAGO NORTE", valor_padrao=0
)

In [342]:
identificacao_crimes_mulher = recriar_regiao_com_valor(
    df=identificacao_crimes_mulher,
    coluna_regiao="ra",
    nome_regiao="VARJÃO",
    valor_padrao=0,
)


In [343]:
colunas_violencia = set(df_violencia_mulher["região_administrativa"])
colunas_crime = set(identificacao_crimes_mulher["ra"])

print("Só no df_violencia:")
print(colunas_femin - colunas_crime)

print("\nSó no df_identidicção do crime:")
print(colunas_crime - colunas_femin)


Só no df_violencia:
set()

Só no df_identidicção do crime:
set()


In [344]:
identificacao_crimes_mulher = identificacao_crimes_mulher.drop("crimes_contra_mulher", axis=1)


In [345]:
identificacao_crimes_mulher.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 252 entries, 0 to 251
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   ano             252 non-null    int32 
 1   ra              252 non-null    object
 2   meio_utilizado  230 non-null    object
 3   local           230 non-null    object
 4   motivação       230 non-null    object
 5   idade___vítima  230 non-null    object
 6   idade___autor   230 non-null    object
dtypes: int32(1), object(6)
memory usage: 12.9+ KB


In [346]:
identificacao_crimes_mulher.rename(columns={"ra": "região_administrativa"}, inplace=True)

In [347]:
identificacao_crimes_mulher.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 252 entries, 0 to 251
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   ano                    252 non-null    int32 
 1   região_administrativa  252 non-null    object
 2   meio_utilizado         230 non-null    object
 3   local                  230 non-null    object
 4   motivação              230 non-null    object
 5   idade___vítima         230 non-null    object
 6   idade___autor          230 non-null    object
dtypes: int32(1), object(6)
memory usage: 12.9+ KB


In [348]:
identificacao_crimes_mulher.describe(include='O')

,região_administrativa,meio_utilizado,local,motivação,idade___vítima,idade___autor
count,252,230,230,230,230,230
unique,33,7,6,5,53,48
top,CEILÂNDIA,Arma Branca,Interior de Residência,Ciúme,35,41
freq,30,117,164,137,11,14


In [349]:
identificacao_crimes_mulher[["idade___vítima", "idade___autor"]] = (
    identificacao_crimes_mulher[["idade___vítima", "idade___autor"]]
    .apply(pd.to_numeric, errors="coerce")
    .astype("Int64")
)


In [350]:
identificacao_crimes_mulher["idade___autor"] = identificacao_crimes_mulher[
    "idade___autor"
].where(identificacao_crimes_mulher["idade___autor"].notna(), 0)


In [351]:
identificacao_crimes_mulher["idade___vítima"] = identificacao_crimes_mulher[
    "idade___vítima"
].where(identificacao_crimes_mulher["idade___vítima"].notna(), 0)


In [352]:
identificacao_crimes_mulher["meio_utilizado"] = identificacao_crimes_mulher[
    "meio_utilizado"
].where(identificacao_crimes_mulher["meio_utilizado"].notna(), "---")


In [353]:
identificacao_crimes_mulher["local"] = identificacao_crimes_mulher[
    "local"
].where(identificacao_crimes_mulher["local"].notna(), "---")


In [354]:
identificacao_crimes_mulher["motivação"] = identificacao_crimes_mulher["motivação"].where(
    identificacao_crimes_mulher["motivação"].notna(), "---"
)


In [355]:
identificacao_crimes_mulher

,ano,região_administrativa,meio_utilizado,local,motivação,idade___vítima,idade___autor
0,2020,VARJÃO,---,---,---,0,0
1,2024,VARJÃO,---,---,---,0,0
2,2023,VARJÃO,---,---,---,0,0
3,2022,VARJÃO,---,---,---,0,0
4,2015,VARJÃO,---,---,---,0,0
...,...,...,...,...,...,...,...
247,2024,VICENTE PIRES,Arma Branca,Interior de Residência,Término Relacionamento,30,29
248,2024,VICENTE PIRES,Arma de Fogo,Interior de Residência,Ciúme,26,32
249,2017,VICENTE PIRES,Arma de Fogo,Interior de Residência,Término Relacionamento,29,39
250,2019,VICENTE PIRES,Arma de Fogo,"Lote vago, terreno abandonado",Ciúme,38,43


In [356]:
identificacao_crimes_mulher.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 252 entries, 0 to 251
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   ano                    252 non-null    int32 
 1   região_administrativa  252 non-null    object
 2   meio_utilizado         252 non-null    object
 3   local                  252 non-null    object
 4   motivação              252 non-null    object
 5   idade___vítima         252 non-null    Int64 
 6   idade___autor          252 non-null    Int64 
dtypes: Int64(2), int32(1), object(4)
memory usage: 13.4+ KB


In [357]:
identificacao_crimes_mulher.describe()

,ano,idade___vítima,idade___autor
count,252.000000,252.0,252.0
mean,2020.380952,32.916667,33.27381
std,3.039860,15.52549,15.760168
min,2015.000000,0.0,0.0
25%,2018.000000,25.0,26.0
50%,2020.000000,34.0,36.0
75%,2023.000000,43.0,42.25
max,2025.000000,94.0,80.0


In [358]:
identificacao_crimes_mulher.describe(include='O')


,região_administrativa,meio_utilizado,local,motivação
count,252,252,252,252
unique,33,8,7,6
top,CEILÂNDIA,Arma Branca,Interior de Residência,Ciúme
freq,30,117,164,137


In [359]:
tabelas


['crimes_contra_mulher',
 'desaparecidos_idade_sexo',
 'desaparecimento_localizados',
 'desaparecimento_regiao',
 'populacao_df',
 'feminicidio',
 'furto_em_veiculo',
 'homicidio',
 'violencia_idosos',
 'violencia_idosos_mensais',
 'violencia_idosos_ocorrencias',
 'violencia_idosos_sexo',
 'injuria_racial',
 'latrocinio',
 'lesao_corporal_morte',
 'populacao_regiao_administrativa',
 'racismo',
 'roubo_comercio',
 'roubo_transporte_coletivo',
 'roubo_veiculo',
 'roubo_pedestre']

In [360]:
df_violencia_idosos = carregar_tabela("violencia_idosos")
df_violencia_idosos

,ranking,regiao_administrativa,jan_ago_2016,jan_ago_2017,variacao_percentual,variacao_absoluta,inserido_em
0,1,GAMA,88,51,-42.05,-37,2026-02-14 03:03:34.094008+00:00
1,2,BRASILIA,21,43,104.76,22,2026-02-14 03:03:34.094008+00:00
2,3,CEILANDIA,27,42,55.56,15,2026-02-14 03:03:34.094008+00:00
3,4,TAGUATINGA,23,30,30.43,7,2026-02-14 03:03:34.094008+00:00
4,5,GUARA,17,15,-11.76,-2,2026-02-14 03:03:34.094008+00:00
5,6,SOBRADINHO,15,15,0.00,0,2026-02-14 03:03:34.094008+00:00
6,7,AGUAS CLARAS,8,11,37.50,3,2026-02-14 03:03:34.094008+00:00
7,8,SAMAMBAIA,11,11,0.00,0,2026-02-14 03:03:34.094008+00:00
8,9,RECANTO DAS EMAS,1,9,800.00,8,2026-02-14 03:03:34.094008+00:00
9,10,SANTA MARIA,23,8,-65.22,-15,2026-02-14 03:03:34.094008+00:00


In [ ]:
df_violencia_idosos_ = df_violencia_idosos.copy()

In [362]:
list(df_violencia_idosos_.columns)

['ranking',
 'regiao_administrativa',
 'jan_ago_2016',
 'jan_ago_2017',
 'variacao_percentual',
 'variacao_absoluta',
 'inserido_em']

In [363]:
df_violencia_idosos_ = df_violencia_idosos_[
    [
        "ranking",
        "regiao_administrativa",
        "jan_ago_2016",
        "jan_ago_2017",
    ]
]

In [364]:
df_violencia_idosos_.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   ranking                31 non-null     int64 
 1   regiao_administrativa  31 non-null     object
 2   jan_ago_2016           31 non-null     int64 
 3   jan_ago_2017           31 non-null     int64 
dtypes: int64(3), object(1)
memory usage: 1.1+ KB


In [365]:
df_violencia_idosos_.describe()

,ranking,jan_ago_2016,jan_ago_2017
count,31.000000,31.000000,31.000000
mean,16.000000,9.354839,9.161290
std,9.092121,16.570151,13.606608
min,1.000000,0.000000,0.000000
25%,8.500000,1.000000,1.500000
50%,16.000000,4.000000,3.000000
75%,23.500000,9.500000,10.000000
max,31.000000,88.000000,51.000000


In [366]:
df_violencia_idosos_.describe(include='O')


,regiao_administrativa
count,31
unique,31
top,GAMA
freq,1


In [367]:
list(df_violencia_idosos['regiao_administrativa'])

['GAMA',
 'BRASILIA',
 'CEILANDIA',
 'TAGUATINGA',
 'GUARA',
 'SOBRADINHO',
 'AGUAS CLARAS',
 'SAMAMBAIA',
 'RECANTO DAS EMAS',
 'SANTA MARIA',
 'VICENTE PIRES',
 'LAGO SUL',
 'PLANALTINA',
 'RIACHO FUNDO',
 'SOBRADINHO 2',
 'SUDOESTE',
 'ITAPOA',
 'BRAZLANDIA',
 'CRUZEIRO',
 'ESTRUTURAL',
 'SAO SEBASTIAO',
 'LAGO NORTE',
 'CANDANGOLANDIA',
 'FERCAL',
 'NUCLEO BANDEIRANTE',
 'RIACHO FUNDO 2',
 'SIA',
 'PARANOA',
 'JARDIM BOTANICO',
 'PARK WAY',
 'VARJAO DO TORTO']

In [368]:
df_violencia_idosos_mensais = carregar_tabela("violencia_idosos_mensais")
df_violencia_idosos_mensais


,ano,mes,mes_num,fato,registro,subnotificacao,inserido_em
0,2016,ABR,4,27,33,6,2026-02-14 03:03:34.136135+00:00
1,2016,AGO,8,35,32,-3,2026-02-14 03:03:34.136135+00:00
2,2016,DEZ,12,27,28,1,2026-02-14 03:03:34.136135+00:00
3,2016,FEV,2,26,31,5,2026-02-14 03:03:34.136135+00:00
4,2016,JAN,1,54,59,5,2026-02-14 03:03:34.136135+00:00
5,2016,JUL,7,31,27,-4,2026-02-14 03:03:34.136135+00:00
6,2016,JUN,6,33,47,14,2026-02-14 03:03:34.136135+00:00
7,2016,MAI,5,35,32,-3,2026-02-14 03:03:34.136135+00:00
8,2016,MAR,3,28,29,1,2026-02-14 03:03:34.136135+00:00
9,2016,NOV,11,20,26,6,2026-02-14 03:03:34.136135+00:00


In [369]:
df_violencia_idosos_mensais_ = df_violencia_idosos_mensais.copy()

In [370]:
list(df_violencia_idosos_mensais_.columns)


['ano', 'mes', 'mes_num', 'fato', 'registro', 'subnotificacao', 'inserido_em']

In [371]:
df_violencia_idosos_mensais_ = df_violencia_idosos_mensais_[['ano', 'mes', 'mes_num', 'fato', 'registro']]

In [372]:
df_violencia_idosos_mensais_.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   ano       24 non-null     int64 
 1   mes       24 non-null     object
 2   mes_num   24 non-null     int64 
 3   fato      24 non-null     int64 
 4   registro  24 non-null     int64 
dtypes: int64(4), object(1)
memory usage: 1.1+ KB


In [373]:
df_violencia_idosos_ocorrencia = carregar_tabela("violencia_idosos_ocorrencias")
df_violencia_idosos_ocorrencia


,ano,ocorrencias,violencia_dentro_de_casa,inserido_em
0,2010,55,22,2026-02-14 03:03:34.178085+00:00
1,2011,63,25,2026-02-14 03:03:34.178085+00:00
2,2012,120,58,2026-02-14 03:03:34.178085+00:00
3,2013,201,91,2026-02-14 03:03:34.178085+00:00
4,2014,321,136,2026-02-14 03:03:34.178085+00:00
5,2015,390,176,2026-02-14 03:03:34.178085+00:00
6,2016,364,154,2026-02-14 03:03:34.178085+00:00


In [374]:

df_violencia_idosos_ocorrencia.columns

Index(['ano', 'ocorrencias', 'violencia_dentro_de_casa', 'inserido_em'], dtype='object')

In [375]:
df_violencia_idosos_ocorrencia_ = df_violencia_idosos_ocorrencia[
    ["ano", "ocorrencias", "violencia_dentro_de_casa"]
]


In [376]:
df_violencia_idosos_ocorrencia_.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 3 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   ano                       7 non-null      int64
 1   ocorrencias               7 non-null      int64
 2   violencia_dentro_de_casa  7 non-null      int64
dtypes: int64(3)
memory usage: 300.0 bytes


In [377]:
df_violencia_idosos_ocorrencia_.describe()

,ano,ocorrencias,violencia_dentro_de_casa
count,7.000000,7.000000,7.000000
mean,2013.000000,216.285714,94.571429
std,2.160247,142.580637,62.364138
min,2010.000000,55.000000,22.000000
25%,2011.500000,91.500000,41.500000
50%,2013.000000,201.000000,91.000000
75%,2014.500000,342.500000,145.000000
max,2016.000000,390.000000,176.000000


In [378]:
df_violencia_idosos_sexo = carregar_tabela("violencia_idosos_sexo")
df_violencia_idosos_sexo

,ano,masculino,feminino,total,inserido_em
0,2010,36,34,70,2026-02-14 03:03:34.212603+00:00
1,2011,32,49,81,2026-02-14 03:03:34.212603+00:00
2,2012,56,89,145,2026-02-14 03:03:34.212603+00:00
3,2013,112,134,246,2026-02-14 03:03:34.212603+00:00
4,2014,179,211,390,2026-02-14 03:03:34.212603+00:00
5,2015,229,268,497,2026-02-14 03:03:34.212603+00:00
6,2016,190,261,451,2026-02-14 03:03:34.212603+00:00


In [379]:
df_violencia_idosos_sexo.columns


Index(['ano', 'masculino', 'feminino', 'total', 'inserido_em'], dtype='object')

In [380]:
df_violencia_idosos_sexo_ = df_violencia_idosos_sexo[['ano', 'masculino', 'feminino']]


In [381]:
df_violencia_idosos_sexo_.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   ano        7 non-null      int64
 1   masculino  7 non-null      int64
 2   feminino   7 non-null      int64
dtypes: int64(3)
memory usage: 300.0 bytes


In [382]:
df_violencia_idosos_sexo_.describe()

,ano,masculino,feminino
count,7.000000,7.000000,7.000000
mean,2013.000000,119.142857,149.428571
std,2.160247,80.846415,97.977646
min,2010.000000,32.000000,34.000000
25%,2011.500000,46.000000,69.000000
50%,2013.000000,112.000000,134.000000
75%,2014.500000,184.500000,236.000000
max,2016.000000,229.000000,268.000000


In [383]:
tabelas

['crimes_contra_mulher',
 'desaparecidos_idade_sexo',
 'desaparecimento_localizados',
 'desaparecimento_regiao',
 'populacao_df',
 'feminicidio',
 'furto_em_veiculo',
 'homicidio',
 'violencia_idosos',
 'violencia_idosos_mensais',
 'violencia_idosos_ocorrencias',
 'violencia_idosos_sexo',
 'injuria_racial',
 'latrocinio',
 'lesao_corporal_morte',
 'populacao_regiao_administrativa',
 'racismo',
 'roubo_comercio',
 'roubo_transporte_coletivo',
 'roubo_veiculo',
 'roubo_pedestre']

In [384]:
df_roubo_pedestre = carregar_tabela("roubo_pedestre")
df_roubo_pedestre

,Região Administrativa,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,inserido_em
0,Distrito Federal,30280.0,38206.0,36755.0,33005.0,29275.0,19717.0,16764.0,16654.0,12781.0,10659.0,2026-02-14 03:03:34.607194+00:00
1,Águas Claras,495.0,544.0,456.0,427.0,197.0,164.0,151.0,101.0,119.0,187.0,2026-02-14 03:03:34.607194+00:00
2,Arniqueira,0.0,0.0,0.0,0.0,0.0,185.0,114.0,120.0,84.0,44.0,2026-02-14 03:03:34.607194+00:00
3,Brasília,2417.0,3490.0,3119.0,3169.0,2975.0,1674.0,1441.0,1722.0,1454.0,991.0,2026-02-14 03:03:34.607194+00:00
4,Brazlândia,369.0,460.0,489.0,332.0,271.0,200.0,136.0,155.0,113.0,61.0,2026-02-14 03:03:34.607194+00:00
5,Candangolândia,82.0,93.0,67.0,62.0,68.0,47.0,44.0,47.0,35.0,25.0,2026-02-14 03:03:34.607194+00:00
6,Ceilândia,5813.0,6202.0,6409.0,6403.0,5693.0,3543.0,3242.0,3549.0,2342.0,2185.0,2026-02-14 03:03:34.607194+00:00
7,Cruzeiro,159.0,119.0,108.0,142.0,121.0,52.0,58.0,52.0,27.0,23.0,2026-02-14 03:03:34.607194+00:00
8,Fercal,7.0,27.0,20.0,30.0,35.0,27.0,9.0,6.0,5.0,11.0,2026-02-14 03:03:34.607194+00:00
9,Gama,1784.0,2040.0,1857.0,1877.0,1690.0,899.0,816.0,706.0,613.0,466.0,2026-02-14 03:03:34.607194+00:00


In [385]:
df_roubo_pedestre.columns

Index(['Região Administrativa', '2015', '2016', '2017', '2018', '2019', '2020',
       '2021', '2022', '2023', '2024', 'inserido_em'],
      dtype='object')

In [386]:
df_roubo_pedestre.drop("inserido_em", axis=1, inplace=True)

In [387]:
df_roubo_pedestre = df_roubo_pedestre.melt(
    id_vars="Região Administrativa",  # coluna fixa
    var_name="ano",  # nome da nova coluna de anos
    value_name="ocorrencia_roubo_pedestre",  # nome da coluna de valores
)


In [388]:
df_roubo_pedestre


,Região Administrativa,ano,ocorrencia_roubo_pedestre
0,Distrito Federal,2015,30280.0
1,Águas Claras,2015,495.0
2,Arniqueira,2015,0.0
3,Brasília,2015,2417.0
4,Brazlândia,2015,369.0
...,...,...,...
345,Sudoeste/Octogonal,2024,23.0
346,Taguatinga,2024,1059.0
347,Varjão,2024,4.0
348,Vicente Pires,2024,141.0


In [389]:
df_roubo_pedestre['Região Administrativa'] = df_roubo_pedestre['Região Administrativa'].str.strip().str.upper()

In [390]:
df_roubo_pedestre = df_roubo_pedestre[df_roubo_pedestre["Região Administrativa"] != "DISTRITO FEDERAL"]

In [391]:
df_roubo_pedestre.info()    

<class 'pandas.core.frame.DataFrame'>
Index: 340 entries, 1 to 349
Data columns (total 3 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Região Administrativa      340 non-null    object 
 1   ano                        340 non-null    object 
 2   ocorrencia_roubo_pedestre  340 non-null    float64
dtypes: float64(1), object(2)
memory usage: 10.6+ KB


In [392]:
df_roubo_pedestre[["ano"]] = (
    df_roubo_pedestre[["ano"]]
    .apply(pd.to_numeric, errors="coerce")
    .astype("Int64")
)


C:\Users\Alexandre\AppData\Local\Temp\ipykernel_21284\4166691039.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_roubo_pedestre[["ano"]] = (


In [393]:
df_roubo_pedestre[["ocorrencia_roubo_pedestre"]] = (
    df_roubo_pedestre[["ocorrencia_roubo_pedestre"]]
    .apply(pd.to_numeric, errors="coerce")
    .astype("Int64")
)


C:\Users\Alexandre\AppData\Local\Temp\ipykernel_21284\3771233894.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_roubo_pedestre[["ocorrencia_roubo_pedestre"]] = (


In [394]:
df_roubo_pedestre.info()


<class 'pandas.core.frame.DataFrame'>
Index: 340 entries, 1 to 349
Data columns (total 3 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Região Administrativa      340 non-null    object
 1   ano                        340 non-null    Int64 
 2   ocorrencia_roubo_pedestre  340 non-null    Int64 
dtypes: Int64(2), object(1)
memory usage: 11.3+ KB


In [395]:
df_roubo_pedestre.describe()

,ano,ocorrencia_roubo_pedestre
count,340.0,340.0
mean,2019.5,717.929412
std,2.876515,1118.484545
min,2015.0,0.0
25%,2017.0,49.5
50%,2019.5,245.0
75%,2022.0,858.75
max,2024.0,6409.0


In [396]:
df_roubo_pedestre.describe(include='O')

,Região Administrativa
count,340
unique,34
top,ÁGUAS CLARAS
freq,10


In [397]:
list(df_roubo_pedestre['Região Administrativa'].unique())

['ÁGUAS CLARAS',
 'ARNIQUEIRA',
 'BRASÍLIA',
 'BRAZLÂNDIA',
 'CANDANGOLÂNDIA',
 'CEILÂNDIA',
 'CRUZEIRO',
 'FERCAL',
 'GAMA',
 'GUARÁ',
 'ITAPOÃ',
 'JARDIM BOTÂNICO',
 'LAGO NORTE',
 'LAGO SUL',
 'NÚCLEO BANDEIRANTE',
 'PARANOÁ',
 'PARK WAY',
 'PLANALTINA',
 'RECANTO DAS EMAS',
 'RIACHO FUNDO',
 'RIACHO FUNDO II',
 'SAMAMBAIA',
 'SANTA MARIA',
 'SÃO SEBASTIÃO',
 'SCIA/ESTRUTURAL',
 'SIA',
 'SOBRADINHO',
 'SOBRADINHO II',
 'SOL NASCENTE/PÔR DO SOL',
 'SUDOESTE/OCTOGONAL',
 'TAGUATINGA',
 'VARJÃO',
 'VICENTE PIRES',
 'ANÁLISE DE 2024']

In [398]:
df_roubo_pedestre = df_roubo_pedestre[df_roubo_pedestre['Região Administrativa'] != 'ANÁLISE DE 2024']

In [399]:
df_roubo_comercio = carregar_tabela("roubo_comercio")
df_roubo_comercio

,Região Administrativa,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,inserido_em
0,Distrito Federal,2630.0,2762.0,2136.0,1774.0,1370.0,904.0,889.0,632.0,525.0,378.0,2026-02-14 03:03:34.468720+00:00
1,Águas Claras,54.0,48.0,19.0,25.0,8.0,16.0,10.0,2.0,4.0,9.0,2026-02-14 03:03:34.468720+00:00
2,Arniqueira,0.0,0.0,0.0,0.0,0.0,13.0,8.0,3.0,3.0,4.0,2026-02-14 03:03:34.468720+00:00
3,Brasília,136.0,111.0,98.0,70.0,71.0,38.0,36.0,34.0,43.0,32.0,2026-02-14 03:03:34.468720+00:00
4,Brazlândia,57.0,36.0,49.0,26.0,14.0,5.0,6.0,6.0,7.0,2.0,2026-02-14 03:03:34.468720+00:00
5,Candangolândia,33.0,19.0,8.0,9.0,7.0,0.0,11.0,3.0,3.0,3.0,2026-02-14 03:03:34.468720+00:00
6,Ceilândia,391.0,398.0,379.0,322.0,227.0,180.0,177.0,100.0,80.0,60.0,2026-02-14 03:03:34.468720+00:00
7,Cruzeiro,12.0,11.0,3.0,4.0,1.0,3.0,3.0,2.0,0.0,1.0,2026-02-14 03:03:34.468720+00:00
8,Fercal,2.0,7.0,2.0,3.0,1.0,2.0,0.0,0.0,1.0,2.0,2026-02-14 03:03:34.468720+00:00
9,Gama,177.0,141.0,133.0,113.0,93.0,61.0,37.0,29.0,24.0,13.0,2026-02-14 03:03:34.468720+00:00


In [400]:
df_roubo_comercio.columns

Index(['Região Administrativa', '2015', '2016', '2017', '2018', '2019', '2020',
       '2021', '2022', '2023', '2024', 'inserido_em'],
      dtype='object')

In [401]:
df_roubo_comercio.drop("inserido_em", axis=1, inplace=True)


In [402]:
df_roubo_comercio = df_roubo_comercio.melt(
    id_vars="Região Administrativa",  # coluna fixa
    var_name="ano",  # nome da nova coluna de anos
    value_name="ocorrencia_roubo_comercio",  # nome da coluna de valores
)

In [403]:
df_roubo_comercio.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 3 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Região Administrativa      350 non-null    object 
 1   ano                        350 non-null    object 
 2   ocorrencia_roubo_comercio  350 non-null    float64
dtypes: float64(1), object(2)
memory usage: 8.3+ KB


In [404]:
df_roubo_comercio[["ano"]] = (
    df_roubo_comercio[["ano"]]
    .apply(pd.to_numeric, errors="coerce").astype("Int64")
)


In [405]:
df_roubo_comercio[["ocorrencia_roubo_comercio"]] = (
    df_roubo_comercio[["ocorrencia_roubo_comercio"]]
    .apply(pd.to_numeric, errors="coerce")
    .astype("Int64")
)


In [406]:
df_roubo_comercio.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 3 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Região Administrativa      350 non-null    object
 1   ano                        350 non-null    Int64 
 2   ocorrencia_roubo_comercio  350 non-null    Int64 
dtypes: Int64(2), object(1)
memory usage: 9.0+ KB


In [407]:
df_roubo_comercio['Região Administrativa'] = df_roubo_comercio['Região Administrativa'].str.strip().str.upper()

In [408]:
list(df_roubo_comercio['Região Administrativa'].unique())

['DISTRITO FEDERAL',
 'ÁGUAS CLARAS',
 'ARNIQUEIRA',
 'BRASÍLIA',
 'BRAZLÂNDIA',
 'CANDANGOLÂNDIA',
 'CEILÂNDIA',
 'CRUZEIRO',
 'FERCAL',
 'GAMA',
 'GUARÁ',
 'ITAPOÃ',
 'JARDIM BOTÂNICO',
 'LAGO NORTE',
 'LAGO SUL',
 'NÚCLEO BANDEIRANTE',
 'PARANOÁ',
 'PARK WAY',
 'PLANALTINA',
 'RECANTO DAS EMAS',
 'RIACHO FUNDO',
 'RIACHO FUNDO II',
 'SAMAMBAIA',
 'SANTA MARIA',
 'SÃO SEBASTIÃO',
 'SCIA/ESTRUTURAL',
 'SIA',
 'SOBRADINHO',
 'SOBRADINHO II',
 'SOL NASCENTE/PÔR DO SOL',
 'SUDOESTE/OCTOGONAL',
 'TAGUATINGA',
 'VARJÃO',
 'VICENTE PIRES',
 'ANÁLISE DE 2024']

In [409]:
df_roubo_comercio = df_roubo_comercio[
    df_roubo_comercio["Região Administrativa"] != "ANÁLISE DE 2024"
]

In [410]:
df_roubo_comercio = df_roubo_comercio[
    df_roubo_comercio["Região Administrativa"] != "DISTRITO FEDERAL"
]


In [411]:
df_roubo_comercio.describe()

,ano,ocorrencia_roubo_comercio
count,330.0,330.0
mean,2019.5,42.424242
std,2.876643,64.986849
min,2015.0,0.0
25%,2017.0,3.0
50%,2019.5,15.5
75%,2022.0,51.5
max,2024.0,398.0


In [412]:
df_roubo_comercio.describe(include='O')

,Região Administrativa
count,330
unique,33
top,ÁGUAS CLARAS
freq,10


In [413]:
df_roubo_transporte_coletivo = carregar_tabela("roubo_transporte_coletivo")
df_roubo_transporte_coletivo

,Região Administrativa,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,inserido_em
0,Distrito Federal,2397.0,3130.0,2672.0,1587.0,1541.0,923.0,640.0,651.0,452.0,229.0,2026-02-14 03:03:34.514115+00:00
1,Águas Claras,16.0,13.0,16.0,12.0,3.0,3.0,3.0,1.0,0.0,1.0,2026-02-14 03:03:34.514115+00:00
2,Arniqueira,0.0,0.0,0.0,0.0,0.0,1.0,2.0,2.0,1.0,1.0,2026-02-14 03:03:34.514115+00:00
3,Brasília,57.0,69.0,59.0,89.0,31.0,42.0,20.0,24.0,18.0,17.0,2026-02-14 03:03:34.514115+00:00
4,Brazlândia,15.0,6.0,23.0,7.0,0.0,1.0,0.0,0.0,1.0,0.0,2026-02-14 03:03:34.514115+00:00
5,Candangolândia,4.0,8.0,10.0,3.0,1.0,6.0,2.0,3.0,2.0,2.0,2026-02-14 03:03:34.514115+00:00
6,Ceilândia,575.0,1026.0,655.0,246.0,268.0,167.0,49.0,100.0,55.0,30.0,2026-02-14 03:03:34.514115+00:00
7,Cruzeiro,0.0,6.0,3.0,5.0,2.0,1.0,1.0,2.0,0.0,0.0,2026-02-14 03:03:34.514115+00:00
8,Fercal,3.0,2.0,8.0,9.0,0.0,2.0,0.0,0.0,4.0,0.0,2026-02-14 03:03:34.514115+00:00
9,Gama,50.0,42.0,28.0,17.0,11.0,10.0,7.0,10.0,11.0,0.0,2026-02-14 03:03:34.514115+00:00


In [414]:
df_roubo_transporte_coletivo.drop("inserido_em", axis=1, inplace=True)

In [415]:
df_roubo_transporte_coletivo = df_roubo_transporte_coletivo.melt(
    id_vars="Região Administrativa",  # coluna fixa
    var_name="ano",  # nome da nova coluna de anos
    value_name="ocorrencia_roubo_transporte_coletivo",  # nome da coluna de valores
)


In [416]:
df_roubo_transporte_coletivo[["ano"]] = (
    df_roubo_transporte_coletivo[["ano"]]
    .apply(pd.to_numeric, errors="coerce")
    .astype("Int64")
)


In [417]:
df_roubo_transporte_coletivo[["ocorrencia_roubo_transporte_coletivo"]] = (
    df_roubo_transporte_coletivo[["ocorrencia_roubo_transporte_coletivo"]]
    .apply(pd.to_numeric, errors="coerce")
    .astype("Int64")
)


In [418]:
df_roubo_transporte_coletivo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 3 columns):
 #   Column                                Non-Null Count  Dtype 
---  ------                                --------------  ----- 
 0   Região Administrativa                 350 non-null    object
 1   ano                                   350 non-null    Int64 
 2   ocorrencia_roubo_transporte_coletivo  350 non-null    Int64 
dtypes: Int64(2), object(1)
memory usage: 9.0+ KB


In [419]:
df_roubo_transporte_coletivo['Região Administrativa'] = df_roubo_transporte_coletivo['Região Administrativa'].str.strip().str.upper()   

In [420]:
list(df_roubo_transporte_coletivo['Região Administrativa'].unique())

['DISTRITO FEDERAL',
 'ÁGUAS CLARAS',
 'ARNIQUEIRA',
 'BRASÍLIA',
 'BRAZLÂNDIA',
 'CANDANGOLÂNDIA',
 'CEILÂNDIA',
 'CRUZEIRO',
 'FERCAL',
 'GAMA',
 'GUARÁ',
 'ITAPOÃ',
 'JARDIM BOTÂNICO',
 'LAGO NORTE',
 'LAGO SUL',
 'NÚCLEO BANDEIRANTE',
 'PARANOÁ',
 'PARK WAY',
 'PLANALTINA',
 'RECANTO DAS EMAS',
 'RIACHO FUNDO',
 'RIACHO FUNDO II',
 'SAMAMBAIA',
 'SANTA MARIA',
 'SÃO SEBASTIÃO',
 'SCIA/ESTRUTURAL',
 'SIA',
 'SOBRADINHO',
 'SOBRADINHO II',
 'SOL NASCENTE/PÔR DO SOL',
 'SUDOESTE/OCTOGONAL',
 'TAGUATINGA',
 'VARJÃO',
 'VICENTE PIRES',
 'ANÁLISE DE 2024']

In [421]:
df_roubo_transporte_coletivo = df_roubo_transporte_coletivo[df_roubo_transporte_coletivo["Região Administrativa"] != "DISTRITO FEDERAL"] 

In [422]:
df_roubo_transporte_coletivo = df_roubo_transporte_coletivo[
    df_roubo_transporte_coletivo["Região Administrativa"] != "ANÁLISE DE 2024"
]


In [423]:
df_roubo_transporte_coletivo.info()


<class 'pandas.core.frame.DataFrame'>
Index: 330 entries, 1 to 348
Data columns (total 3 columns):
 #   Column                                Non-Null Count  Dtype 
---  ------                                --------------  ----- 
 0   Região Administrativa                 330 non-null    object
 1   ano                                   330 non-null    Int64 
 2   ocorrencia_roubo_transporte_coletivo  330 non-null    Int64 
dtypes: Int64(2), object(1)
memory usage: 11.0+ KB


In [424]:
df_roubo_transporte_coletivo.describe()

,ano,ocorrencia_roubo_transporte_coletivo
count,330.0,330.0
mean,2019.5,43.09697
std,2.876643,112.563159
min,2015.0,0.0
25%,2017.0,2.0
50%,2019.5,7.0
75%,2022.0,24.75
max,2024.0,1026.0


In [425]:
df_roubo_veiculo = carregar_tabela('roubo_veiculo')
df_roubo_veiculo

,Região Administrativa,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,inserido_em
0,Distrito Federal,4808.0,5663.0,4851.0,3993.0,3424.0,2218.0,2033.0,1553.0,1291.0,1018.0,2026-02-14 03:03:34.562131+00:00
1,Águas Claras,105.0,103.0,83.0,48.0,24.0,14.0,19.0,9.0,13.0,18.0,2026-02-14 03:03:34.562131+00:00
2,Arniqueira,0.0,0.0,0.0,0.0,0.0,30.0,19.0,20.0,12.0,6.0,2026-02-14 03:03:34.562131+00:00
3,Brasília,303.0,294.0,351.0,262.0,134.0,78.0,56.0,45.0,45.0,32.0,2026-02-14 03:03:34.562131+00:00
4,Brazlândia,85.0,71.0,81.0,74.0,55.0,39.0,34.0,26.0,24.0,18.0,2026-02-14 03:03:34.562131+00:00
5,Candangolândia,14.0,14.0,13.0,8.0,9.0,9.0,1.0,3.0,2.0,1.0,2026-02-14 03:03:34.562131+00:00
6,Ceilândia,853.0,871.0,832.0,714.0,611.0,368.0,379.0,311.0,232.0,179.0,2026-02-14 03:03:34.562131+00:00
7,Cruzeiro,39.0,30.0,38.0,10.0,12.0,3.0,5.0,4.0,0.0,1.0,2026-02-14 03:03:34.562131+00:00
8,Fercal,6.0,13.0,9.0,13.0,14.0,2.0,1.0,8.0,4.0,5.0,2026-02-14 03:03:34.562131+00:00
9,Gama,224.0,355.0,259.0,194.0,177.0,129.0,94.0,93.0,55.0,52.0,2026-02-14 03:03:34.562131+00:00


In [426]:
df_roubo_veiculo.drop("inserido_em", axis=1, inplace=True)  

In [427]:
df_roubo_veiculo =df_roubo_veiculo.melt(
    id_vars="Região Administrativa",  # coluna fixa
    var_name="ano",  # nome da nova coluna de anos
    value_name="ocorrencia_roubo_veiculo",  # nome da coluna de valores
)   

In [428]:
df_roubo_veiculo[["ano"]] = (
    df_roubo_veiculo[["ano"]].apply(pd.to_numeric, errors="coerce").astype("Int64")
)


In [429]:
df_roubo_veiculo[["ocorrencia_roubo_veiculo"]] = (
    df_roubo_veiculo[["ocorrencia_roubo_veiculo"]]
    .apply(pd.to_numeric, errors="coerce")
    .astype("Int64")
)


In [430]:
df_roubo_veiculo[ 'Região Administrativa'] = df_roubo_veiculo[ 'Região Administrativa'].str.strip().str.upper()

In [431]:
list(df_roubo_veiculo['Região Administrativa'].unique()                                         )

['DISTRITO FEDERAL',
 'ÁGUAS CLARAS',
 'ARNIQUEIRA',
 'BRASÍLIA',
 'BRAZLÂNDIA',
 'CANDANGOLÂNDIA',
 'CEILÂNDIA',
 'CRUZEIRO',
 'FERCAL',
 'GAMA',
 'GUARÁ',
 'ITAPOÃ',
 'JARDIM BOTÂNICO',
 'LAGO NORTE',
 'LAGO SUL',
 'NÚCLEO BANDEIRANTE',
 'PARANOÁ',
 'PARK WAY',
 'PLANALTINA',
 'RECANTO DAS EMAS',
 'RIACHO FUNDO',
 'RIACHO FUNDO II',
 'SAMAMBAIA',
 'SANTA MARIA',
 'SÃO SEBASTIÃO',
 'SCIA/ESTRUTURAL',
 'SIA',
 'SOBRADINHO',
 'SOBRADINHO II',
 'SOL NASCENTE/PÔR DO SOL',
 'SUDOESTE/OCTOGONAL',
 'TAGUATINGA',
 'VARJÃO',
 'VICENTE PIRES']

In [432]:
df_roubo_veiculo = df_roubo_veiculo[df_roubo_veiculo["Região Administrativa"] != "DISTRITO FEDERAL"]    

In [433]:
df_roubo_veiculo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 330 entries, 1 to 339
Data columns (total 3 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Região Administrativa     330 non-null    object
 1   ano                       330 non-null    Int64 
 2   ocorrencia_roubo_veiculo  330 non-null    Int64 
dtypes: Int64(2), object(1)
memory usage: 11.0+ KB


In [434]:
df_roubo_veiculo.describe()

,ano,ocorrencia_roubo_veiculo
count,330.0,330.0
mean,2019.5,93.490909
std,2.876643,161.491928
min,2015.0,0.0
25%,2017.0,11.25
50%,2019.5,30.0
75%,2022.0,92.5
max,2024.0,914.0


In [435]:
df_furto_em_veiculo = carregar_tabela("furto_em_veiculo")
df_furto_em_veiculo

,Região Administrativa,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,inserido_em
0,Águas Claras,1008,768,603,614,501,370,423,475,408,359,2026-02-14 03:03:34.003702+00:00
1,Arniqueira,0,0,0,0,0,54,46,62,72,30,2026-02-14 03:03:34.003702+00:00
2,Brasília,3297,4506,4883,3771,2777,1718,1996,3068,2750,2572,2026-02-14 03:03:34.003702+00:00
3,Brazlândia,96,101,114,101,90,70,52,78,47,45,2026-02-14 03:03:34.003702+00:00
4,Candangolândia,33,31,26,30,22,23,14,14,11,3,2026-02-14 03:03:34.003702+00:00
5,Ceilândia,1114,1280,1081,986,1213,767,590,627,640,644,2026-02-14 03:03:34.003702+00:00
6,Cruzeiro,136,87,74,92,67,36,31,34,50,54,2026-02-14 03:03:34.003702+00:00
7,Fercal,6,4,23,18,13,11,8,7,7,4,2026-02-14 03:03:34.003702+00:00
8,Gama,516,434,411,389,384,279,308,295,309,216,2026-02-14 03:03:34.003702+00:00
9,Guará,717,664,564,433,333,212,314,393,231,290,2026-02-14 03:03:34.003702+00:00


In [436]:
df_furto_em_veiculo.drop("inserido_em", axis=1, inplace=True)


In [437]:
df_furto_em_veiculo = df_furto_em_veiculo.melt(
    id_vars="Região Administrativa",  # coluna fixa
    var_name="ano",  # nome da coluna com os nomes das variáveis
    value_name="ocorrencia_furto_veículo"  # nome da coluna com os valores
)   

In [438]:
df_furto_em_veiculo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 330 entries, 0 to 329
Data columns (total 3 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Região Administrativa     330 non-null    object
 1   ano                       330 non-null    object
 2   ocorrencia_furto_veículo  330 non-null    int64 
dtypes: int64(1), object(2)
memory usage: 7.9+ KB


In [439]:
df_furto_em_veiculo['Região Administrativa'] = df_furto_em_veiculo['Região Administrativa'].str.strip().str.upper()

In [440]:
df_furto_em_veiculo[["ano"]] = (
    df_furto_em_veiculo[["ano"]].apply(pd.to_numeric, errors="coerce").astype("Int64")
)


In [441]:
list(df_furto_em_veiculo['Região Administrativa'].unique())


['ÁGUAS CLARAS',
 'ARNIQUEIRA',
 'BRASÍLIA',
 'BRAZLÂNDIA',
 'CANDANGOLÂNDIA',
 'CEILÂNDIA',
 'CRUZEIRO',
 'FERCAL',
 'GAMA',
 'GUARÁ',
 'ITAPOÃ',
 'JARDIM BOTÂNICO',
 'LAGO NORTE',
 'LAGO SUL',
 'NÚCLEO BANDEIRANTE',
 'PARANOÁ',
 'PARK WAY',
 'PLANALTINA',
 'RECANTO DAS EMAS',
 'RIACHO FUNDO',
 'RIACHO FUNDO II',
 'SAMAMBAIA',
 'SANTA MARIA',
 'SÃO SEBASTIÃO',
 'SCIA/ESTRUTURAL',
 'SIA',
 'SOBRADINHO',
 'SOBRADINHO II',
 'SOL NASCENTE/PÔR DO SOL',
 'SUDOESTE/OCTOGONAL',
 'TAGUATINGA',
 'VARJÃO',
 'VICENTE PIRES']

In [442]:
df_furto_em_veiculo.describe()

,ano,ocorrencia_furto_veículo
count,330.0,330.000000
mean,2019.5,278.124242
std,2.876643,591.577218
min,2015.0,0.000000
25%,2017.0,41.500000
50%,2019.5,96.000000
75%,2022.0,232.500000
max,2024.0,4883.000000


In [443]:
colunas_roubo_comercio = set(df_roubo_comercio["Região Administrativa"])
colunas_roubo_pedeste = set(df_roubo_pedestre["Região Administrativa"])
colunas_roubo_transporte_coletivo = set(df_roubo_transporte_coletivo["Região Administrativa"])
colunas_roubo_veiculo = set(df_roubo_veiculo["Região Administrativa"])
colunas_furto_em_veiculo = set(df_furto_em_veiculo["Região Administrativa"])

print("Só no df_roubo de veiculo:")
print(colunas_roubo_veiculo - colunas_furto_em_veiculo)

print("\nSó no df_roubo de furto em veículos:")
print(colunas_roubo_veiculo - colunas_furto_em_veiculo)


Só no df_roubo de veiculo:
set()

Só no df_roubo de furto em veículos:
set()


In [444]:
df_rubo_1 = pd.merge(
    df_roubo_comercio,
    df_roubo_pedestre,
    on=["ano", "Região Administrativa"],
    how="outer",
)


In [445]:
df_rubo_2 = pd.merge(
    df_rubo_1,
    df_roubo_transporte_coletivo,
    on=["ano", "Região Administrativa"],
    how="outer",
)       


In [446]:
df_roubo_3 = pd.merge(
    df_rubo_2,
    df_roubo_veiculo,
    on=["ano", "Região Administrativa"],
    how="outer",
)   

In [447]:
df_roubo_furto = pd.merge(  
    df_roubo_3, 
    df_furto_em_veiculo,
    on=["ano", "Região Administrativa"],
    how="outer",
)    

In [448]:
df_roubo_furto.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 330 entries, 0 to 329
Data columns (total 7 columns):
 #   Column                                Non-Null Count  Dtype 
---  ------                                --------------  ----- 
 0   Região Administrativa                 330 non-null    object
 1   ano                                   330 non-null    Int64 
 2   ocorrencia_roubo_comercio             330 non-null    Int64 
 3   ocorrencia_roubo_pedestre             330 non-null    Int64 
 4   ocorrencia_roubo_transporte_coletivo  330 non-null    Int64 
 5   ocorrencia_roubo_veiculo              330 non-null    Int64 
 6   ocorrencia_furto_veículo              330 non-null    int64 
dtypes: Int64(5), int64(1), object(1)
memory usage: 19.8+ KB


In [449]:
df_roubo_furto.describe()

,ano,ocorrencia_roubo_comercio,ocorrencia_roubo_pedestre,ocorrencia_roubo_transporte_coletivo,ocorrencia_roubo_veiculo,ocorrencia_furto_veículo
count,330.0,330.0,330.0,330.0,330.0,330.000000
mean,2019.5,42.424242,739.684848,43.09697,93.490909,278.124242
std,2.876643,64.986849,1128.224777,112.563159,161.491928,591.577218
min,2015.0,0.0,0.0,0.0,0.0,0.000000
25%,2017.0,3.0,63.0,2.0,11.25,41.500000
50%,2019.5,15.5,270.5,7.0,30.0,96.000000
75%,2022.0,51.5,894.0,24.75,92.5,232.500000
max,2024.0,398.0,6409.0,1026.0,914.0,4883.000000


In [450]:
df_homicidio = carregar_tabela("homicidio")
df_homicidio


,regiao_administrativa,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,inserido_em
0,Distrito Federal,619.0,582.0,493.0,434.0,386.0,357.0,287.0,261.0,234.0,207.0,2026-02-14 03:03:34.046375+00:00
1,Águas Claras,11.0,3.0,6.0,6.0,8.0,3.0,4.0,0.0,1.0,1.0,2026-02-14 03:03:34.046375+00:00
2,Arniqueira,NaN,NaN,NaN,NaN,NaN,6.0,4.0,4.0,4.0,0.0,2026-02-14 03:03:34.046375+00:00
3,Brasília (Plano Piloto),23.0,14.0,16.0,19.0,15.0,9.0,7.0,7.0,6.0,8.0,2026-02-14 03:03:34.046375+00:00
4,Brazlândia,21.0,19.0,11.0,21.0,2.0,5.0,9.0,7.0,5.0,5.0,2026-02-14 03:03:34.046375+00:00
5,Candangolândia,6.0,2.0,5.0,0.0,0.0,2.0,1.0,3.0,1.0,0.0,2026-02-14 03:03:34.046375+00:00
6,Ceilândia,112.0,90.0,77.0,83.0,77.0,46.0,42.0,31.0,41.0,33.0,2026-02-14 03:03:34.046375+00:00
7,Cruzeiro,2.0,0.0,0.0,0.0,2.0,0.0,1.0,1.0,0.0,0.0,2026-02-14 03:03:34.046375+00:00
8,Fercal,3.0,5.0,7.0,5.0,5.0,4.0,3.0,4.0,2.0,3.0,2026-02-14 03:03:34.046375+00:00
9,Gama,38.0,40.0,38.0,25.0,17.0,16.0,12.0,6.0,8.0,8.0,2026-02-14 03:03:34.046375+00:00


In [451]:
df_homicidio.drop("inserido_em", axis=1, inplace=True)

In [452]:
df_homicidio = df_homicidio.melt(
    id_vars="regiao_administrativa",  # coluna fixa
    var_name="ano",  # nome da nova coluna com os valores das colunas originais
    value_name="homicidios"  # nome da nova coluna com os valores
)

In [453]:
df_homicidio.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 3 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   regiao_administrativa  350 non-null    object 
 1   ano                    350 non-null    object 
 2   homicidios             332 non-null    float64
dtypes: float64(1), object(2)
memory usage: 8.3+ KB


In [454]:
df_homicidio["homicidios"].fillna(0, inplace=True)


C:\Users\Alexandre\AppData\Local\Temp\ipykernel_21284\3761192478.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_homicidio["homicidios"].fillna(0, inplace=True)


In [455]:
df_homicidio['homicidios'].isnull().sum()

np.int64(0)

In [456]:
df_homicidio["regiao_administrativa"] = (
    df_homicidio["regiao_administrativa"].str.strip().str.upper()
)


In [457]:
list(df_homicidio["regiao_administrativa"].unique())

['DISTRITO FEDERAL',
 'ÁGUAS CLARAS',
 'ARNIQUEIRA',
 'BRASÍLIA (PLANO PILOTO)',
 'BRAZLÂNDIA',
 'CANDANGOLÂNDIA',
 'CEILÂNDIA',
 'CRUZEIRO',
 'FERCAL',
 'GAMA',
 'GUARÁ',
 'ITAPOÃ',
 'JARDIM BOTÂNICO',
 'LAGO NORTE',
 'LAGO SUL',
 'NÚCLEO BANDEIRANTE',
 'PARANOÁ',
 'PARK WAY',
 'PLANALTINA',
 'RECANTO DAS EMAS',
 'RIACHO FUNDO',
 'RIACHO FUNDO II',
 'SAMAMBAIA',
 'SANTA MARIA',
 'SÃO SEBASTIÃO',
 'SCIA/ESTRUTURAL',
 'SIA',
 'SOBRADINHO',
 'SOBRADINHO II',
 'SOL NASCENTE/PÔR DO SOL',
 'SUDOESTE/OCTOGONAL',
 'TAGUATINGA',
 'VARJÃO',
 'VICENTE PIRES',
 'UNIDADES PRISIONAIS']

In [458]:
df_homicidio = df_homicidio[df_homicidio['regiao_administrativa'] != "DISTRITO FEDERAL"]

In [459]:
df_homicidio.info()

<class 'pandas.core.frame.DataFrame'>
Index: 340 entries, 1 to 349
Data columns (total 3 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   regiao_administrativa  340 non-null    object 
 1   ano                    340 non-null    object 
 2   homicidios             340 non-null    float64
dtypes: float64(1), object(2)
memory usage: 10.6+ KB


In [460]:
df_homicidio[["ano"]] = (
    df_homicidio[["ano"]].apply(pd.to_numeric, errors="coerce").astype("Int64")
)


In [461]:
df_homicidio[["homicidios"]] = (
    df_homicidio[["homicidios"]].apply(pd.to_numeric, errors="coerce").astype("Int64")
)


In [462]:
df_homicidio.rename(columns={"regiao_administrativa": "regiao"}, inplace=True)


In [463]:
df_homicidio


,regiao,ano,homicidios
1,ÁGUAS CLARAS,2015,11
2,ARNIQUEIRA,2015,0
3,BRASÍLIA (PLANO PILOTO),2015,23
4,BRAZLÂNDIA,2015,21
5,CANDANGOLÂNDIA,2015,6
...,...,...,...
345,SUDOESTE/OCTOGONAL,2024,0
346,TAGUATINGA,2024,10
347,VARJÃO,2024,0
348,VICENTE PIRES,2024,5


In [464]:
df_homicidio.describe()

,ano,homicidios
count,340.0,340.0
mean,2019.5,11.352941
std,2.876515,15.515493
min,2015.0,0.0
25%,2017.0,1.0
50%,2019.5,5.0
75%,2022.0,16.0
max,2024.0,112.0


In [465]:
df_latrocinio = carregar_tabela("latrocinio")
df_latrocinio

,regiao,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,inserido_em
0,Distrito Federal,46,47,36,32,25,32,23,20,18,8,2026-02-14 03:03:34.290958+00:00
1,Águas Claras,0,0,1,0,0,1,0,0,0,0,2026-02-14 03:03:34.290958+00:00
2,Arniqueira,0,0,0,0,0,1,0,1,1,0,2026-02-14 03:03:34.290958+00:00
3,Brasília (Plano Piloto),1,0,4,1,1,4,0,0,0,0,2026-02-14 03:03:34.290958+00:00
4,Brazlândia,0,1,3,1,1,1,0,1,0,1,2026-02-14 03:03:34.290958+00:00
5,Candangolândia,1,0,1,0,0,0,0,0,0,0,2026-02-14 03:03:34.290958+00:00
6,Ceilândia,6,3,8,7,5,3,3,7,1,1,2026-02-14 03:03:34.290958+00:00
7,Cruzeiro,1,2,0,0,0,0,0,0,0,0,2026-02-14 03:03:34.290958+00:00
8,Fercal,0,0,1,0,0,0,0,0,2,1,2026-02-14 03:03:34.290958+00:00
9,Gama,3,3,1,3,1,1,2,0,0,0,2026-02-14 03:03:34.290958+00:00


In [466]:
df_latrocinio.drop("inserido_em", axis=1, inplace=True)

In [467]:
df_latrocinio = df_latrocinio.melt(
    id_vars="regiao",  # coluna fixa
    var_name="ano",  # nome da coluna que irá conter os valores das colunas originais
    value_name="latrocinios"  # nome da coluna que irá conter os valores das colunas originais
)

In [468]:
df_latrocinio.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 340 entries, 0 to 339
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   regiao       340 non-null    object
 1   ano          340 non-null    object
 2   latrocinios  340 non-null    int64 
dtypes: int64(1), object(2)
memory usage: 8.1+ KB


In [469]:
df_latrocinio[["ano"]] = (
    df_latrocinio[["ano"]].apply(pd.to_numeric, errors="coerce").astype("Int64")
)


In [470]:
df_latrocinio[["latrocinios"]] = (
    df_latrocinio[["latrocinios"]].apply(pd.to_numeric, errors="coerce").astype("Int64")
)


In [471]:
df_latrocinio["regiao"] = df_latrocinio["regiao"].str.strip().str.upper()

In [472]:
df_latrocinio

,regiao,ano,latrocinios
0,DISTRITO FEDERAL,2015,46
1,ÁGUAS CLARAS,2015,0
2,ARNIQUEIRA,2015,0
3,BRASÍLIA (PLANO PILOTO),2015,1
4,BRAZLÂNDIA,2015,0
...,...,...,...
335,SOL NASCENTE/PÔR DO SOL,2024,0
336,SUDOESTE/OCTOGONAL,2024,0
337,TAGUATINGA,2024,2
338,VARJÃO,2024,0


In [473]:
list(df_latrocinio["regiao"].unique())

['DISTRITO FEDERAL',
 'ÁGUAS CLARAS',
 'ARNIQUEIRA',
 'BRASÍLIA (PLANO PILOTO)',
 'BRAZLÂNDIA',
 'CANDANGOLÂNDIA',
 'CEILÂNDIA',
 'CRUZEIRO',
 'FERCAL',
 'GAMA',
 'GUARÁ',
 'ITAPOÃ',
 'JARDIM BOTÂNICO',
 'LAGO NORTE',
 'LAGO SUL',
 'NÚCLEO BANDEIRANTE',
 'PARANOÁ',
 'PARK WAY',
 'PLANALTINA',
 'RECANTO DAS EMAS',
 'RIACHO FUNDO',
 'RIACHO FUNDO II',
 'SAMAMBAIA',
 'SANTA MARIA',
 'SÃO SEBASTIÃO',
 'SCIA/ESTRUTURAL',
 'SIA',
 'SOBRADINHO',
 'SOBRADINHO II',
 'SOL NASCENTE/PÔR DO SOL',
 'SUDOESTE/OCTOGONAL',
 'TAGUATINGA',
 'VARJÃO',
 'VICENTE PIRES']

In [474]:
df_latrocinio = df_latrocinio[df_latrocinio["regiao"] != "DISTRITO FEDERAL"]


In [475]:
df_latrocinio = recriar_regiao_com_valor(
    df=df_latrocinio,
    coluna_regiao="regiao",
    nome_regiao="UNIDADES PRISIONAIS",
    coluna_valor="latrocinios",
    valor_padrao=0,
)

In [476]:
df_latrocinio.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 340 entries, 0 to 339
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ano          340 non-null    Int64 
 1   regiao       340 non-null    object
 2   latrocinios  340 non-null    Int64 
dtypes: Int64(2), object(1)
memory usage: 8.8+ KB


In [477]:
df_latrocinio.describe()

,ano,latrocinios
count,340.0,340.0
mean,2019.5,0.844118
std,2.876515,1.425369
min,2015.0,0.0
25%,2017.0,0.0
50%,2019.5,0.0
75%,2022.0,1.0
max,2024.0,8.0


In [478]:
df_lesao_morte = carregar_tabela("lesao_corporal_morte")
df_lesao_morte

,regiao,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,inserido_em
0,Distrito Federal,6,5,5,8,6,5,7,5,1,13,2026-02-14 03:03:34.337616+00:00
1,Águas Claras,0,0,0,0,0,0,0,0,0,0,2026-02-14 03:03:34.337616+00:00
2,Arniqueira,0,0,0,0,0,0,0,1,0,0,2026-02-14 03:03:34.337616+00:00
3,Brasília (Plano Piloto),0,1,0,0,0,1,0,0,0,1,2026-02-14 03:03:34.337616+00:00
4,Brazlândia,0,0,1,2,0,0,0,0,0,1,2026-02-14 03:03:34.337616+00:00
5,Candangolândia,0,0,0,0,0,0,0,0,0,0,2026-02-14 03:03:34.337616+00:00
6,Ceilândia,0,1,0,2,1,0,1,1,0,1,2026-02-14 03:03:34.337616+00:00
7,Cruzeiro,0,0,0,0,0,0,0,0,0,0,2026-02-14 03:03:34.337616+00:00
8,Fercal,0,0,0,0,0,0,0,0,0,0,2026-02-14 03:03:34.337616+00:00
9,Gama,0,0,0,0,0,0,0,0,1,1,2026-02-14 03:03:34.337616+00:00


In [479]:
df_lesao_morte.drop("inserido_em", axis=1, inplace=True)

In [480]:
df_lesao_morte = df_lesao_morte.melt(
    id_vars="regiao",  # coluna fixa         
    var_name="ano",  # nome da nova coluna com os valores das colunas originais
    value_name="lesao_corporal_morte"  # nome da nova coluna com os valores das colunas originais
)

In [481]:
df_lesao_morte.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 340 entries, 0 to 339
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   regiao                340 non-null    object
 1   ano                   340 non-null    object
 2   lesao_corporal_morte  340 non-null    int64 
dtypes: int64(1), object(2)
memory usage: 8.1+ KB


In [482]:
df_lesao_morte[["ano"]] = (
    df_lesao_morte[["ano"]].apply(pd.to_numeric, errors="coerce").astype("Int64")
)


In [483]:
df_lesao_morte[["lesao_corporal_morte"]] = (
    df_lesao_morte[["lesao_corporal_morte"]]
    .apply(pd.to_numeric, errors="coerce")
    .astype("Int64")
)


In [484]:
list(df_lesao_morte["regiao"].unique())

['Distrito Federal',
 'Águas Claras',
 'Arniqueira',
 'Brasília (Plano Piloto)',
 'Brazlândia',
 'Candangolândia',
 'Ceilândia',
 'Cruzeiro',
 'Fercal',
 'Gama',
 'Guará',
 'Itapoã',
 'Jardim Botânico',
 'Lago Norte',
 'Lago Sul',
 'Núcleo Bandeirante',
 'Paranoá',
 'Park Way',
 'Planaltina',
 'Recanto das Emas',
 'Riacho Fundo',
 'Riacho Fundo II',
 'Samambaia',
 'Santa Maria',
 'São Sebastião',
 'SCIA/Estrutural',
 'SIA',
 'Sobradinho',
 'Sobradinho II',
 'Sol Nascente/Pôr do Sol',
 'Sudoeste/Octogonal',
 'Taguatinga',
 'Varjão',
 'Vicente Pires']

In [485]:
df_lesao_morte["regiao"] = df_lesao_morte["regiao"].str.strip().str.upper()


In [486]:
df_lesao_morte = df_lesao_morte[df_lesao_morte["regiao"] != "DISTRITO FEDERAL"]


In [487]:
df_lesao_morte = recriar_regiao_com_valor(
    df=df_lesao_morte,
    coluna_regiao="regiao",
    nome_regiao="UNIDADES PRISIONAIS",
    coluna_valor="lesao_corporal_morte",
    valor_padrao=0,
)

In [488]:
df_lesao_morte.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 340 entries, 0 to 339
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   ano                   340 non-null    Int64 
 1   regiao                340 non-null    object
 2   lesao_corporal_morte  340 non-null    Int64 
dtypes: Int64(2), object(1)
memory usage: 8.8+ KB


In [489]:
df_lesao_morte.describe()

,ano,lesao_corporal_morte
count,340.0,340.0
mean,2019.5,0.179412
std,2.876515,0.526736
min,2015.0,0.0
25%,2017.0,0.0
50%,2019.5,0.0
75%,2022.0,0.0
max,2024.0,4.0


In [490]:
tabelas

['crimes_contra_mulher',
 'desaparecidos_idade_sexo',
 'desaparecimento_localizados',
 'desaparecimento_regiao',
 'populacao_df',
 'feminicidio',
 'furto_em_veiculo',
 'homicidio',
 'violencia_idosos',
 'violencia_idosos_mensais',
 'violencia_idosos_ocorrencias',
 'violencia_idosos_sexo',
 'injuria_racial',
 'latrocinio',
 'lesao_corporal_morte',
 'populacao_regiao_administrativa',
 'racismo',
 'roubo_comercio',
 'roubo_transporte_coletivo',
 'roubo_veiculo',
 'roubo_pedestre']

In [491]:
colunas_homicido = set(df_homicidio["regiao"])
colunas_latrocinio = set(df_latrocinio["regiao"])
colunas_lesao_morte = set(df_lesao_morte["regiao"])

print("Só no df_homicidio:")
print(colunas_homicido - colunas_latrocinio)

print("\nSó no df_lesao_morte:")
print(colunas_lesao_morte - colunas_homicido)

print("\nSó no df_latrocinio:")
print(colunas_latrocinio - colunas_homicido)

Só no df_homicidio:
set()

Só no df_lesao_morte:
set()

Só no df_latrocinio:
set()


In [492]:
df_hom_ = pd.merge(
    df_homicidio,
    df_latrocinio,
    on=["ano", "regiao"],
    how="outer",
)


In [493]:
df_homicidios = pd.merge(
    df_hom_,
    df_lesao_morte,
    on=["ano", "regiao"],
    how="outer",    
)

In [494]:
df_homicidios

,regiao,ano,homicidios,latrocinios,lesao_corporal_morte
0,ARNIQUEIRA,2015,0,0,0
1,BRASÍLIA (PLANO PILOTO),2015,23,1,0
2,BRAZLÂNDIA,2015,21,0,0
3,CANDANGOLÂNDIA,2015,6,1,0
4,CEILÂNDIA,2015,112,6,0
...,...,...,...,...,...
335,TAGUATINGA,2024,10,2,3
336,UNIDADES PRISIONAIS,2024,5,0,0
337,VARJÃO,2024,0,0,0
338,VICENTE PIRES,2024,5,1,0


In [495]:
df_racismo = carregar_tabela("racismo")
df_racismo

,regiao,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,inserido_em
0,Distrito Federal,4,10,1,5,3,11,16,28,41,39,2026-02-14 03:03:34.427041+00:00
1,Águas Claras,0,0,0,0,0,1,3,1,1,1,2026-02-14 03:03:34.427041+00:00
2,Arniqueira,0,0,0,0,0,0,0,0,0,0,2026-02-14 03:03:34.427041+00:00
3,Brasília,1,4,1,1,1,4,1,13,20,17,2026-02-14 03:03:34.427041+00:00
4,Brazlândia,0,1,0,0,0,0,0,1,2,0,2026-02-14 03:03:34.427041+00:00
5,Candangolândia,0,0,0,0,0,0,0,1,0,2,2026-02-14 03:03:34.427041+00:00
6,Ceilândia,1,3,0,0,0,1,1,1,5,1,2026-02-14 03:03:34.427041+00:00
7,Cruzeiro,0,0,0,0,0,0,0,1,0,0,2026-02-14 03:03:34.427041+00:00
8,Fercal,0,0,0,0,0,0,0,0,0,0,2026-02-14 03:03:34.427041+00:00
9,Gama,0,0,0,0,0,0,1,0,1,1,2026-02-14 03:03:34.427041+00:00


In [496]:
df_racismo.drop("inserido_em", axis=1, inplace=True)

In [497]:
df_racismo = df_racismo.melt(
    id_vars="regiao",  # coluna fixa
    var_name="ano",  # nome da coluna que irá conter os valores das colunas originais
    value_name="racismo"  # nome da coluna que irá conter os valores das colunas originais
)

In [498]:
df_racismo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 340 entries, 0 to 339
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   regiao   340 non-null    object
 1   ano      340 non-null    object
 2   racismo  340 non-null    int64 
dtypes: int64(1), object(2)
memory usage: 8.1+ KB


In [499]:
df_racismo[["ano"]] = (
    df_racismo[["ano"]].apply(pd.to_numeric, errors="coerce").astype("Int64")
)


In [500]:
list(df_racismo["regiao"].unique())

['Distrito Federal',
 'Águas Claras',
 'Arniqueira',
 'Brasília',
 'Brazlândia',
 'Candangolândia',
 'Ceilândia',
 'Cruzeiro',
 'Fercal',
 'Gama',
 'Guará',
 'Itapoã',
 'Jardim Botânico',
 'Lago Norte',
 'Lago Sul',
 'Núcleo Bandeirante',
 'Paranoá',
 'Park Way',
 'Planaltina',
 'Recanto das Emas',
 'Riacho Fundo',
 'Riacho Fundo II',
 'Samambaia',
 'Santa Maria',
 'São Sebastião',
 'SCIA/Estrutural',
 'SIA',
 'Sobradinho',
 'Sobradinho II',
 'Sol Nascente/Pôr do Sol',
 'Sudoeste/Octogonal',
 'Taguatinga',
 'Varjão',
 'Vicente Pires']

In [501]:
df_racismo["regiao"] = df_racismo["regiao"].str.strip().str.upper()


In [502]:
df_racismo = df_racismo[df_racismo["regiao"] != "DISTRITO FEDERAL"]
df_racismo

,regiao,ano,racismo
1,ÁGUAS CLARAS,2015,0
2,ARNIQUEIRA,2015,0
3,BRASÍLIA,2015,1
4,BRAZLÂNDIA,2015,0
5,CANDANGOLÂNDIA,2015,0
...,...,...,...
335,SOL NASCENTE/PÔR DO SOL,2024,0
336,SUDOESTE/OCTOGONAL,2024,2
337,TAGUATINGA,2024,2
338,VARJÃO,2024,0


In [503]:
df_racismo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 330 entries, 1 to 339
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   regiao   330 non-null    object
 1   ano      330 non-null    Int64 
 2   racismo  330 non-null    int64 
dtypes: Int64(1), int64(1), object(1)
memory usage: 10.6+ KB


In [504]:
df_racismo.describe()

,ano,racismo
count,330.0,330.000000
mean,2019.5,0.478788
std,2.876643,1.729725
min,2015.0,0.000000
25%,2017.0,0.000000
50%,2019.5,0.000000
75%,2022.0,0.000000
max,2024.0,20.000000


In [505]:
df_injuria =carregar_tabela('injuria_racial')
df_injuria

,regiao,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,inserido_em
0,Distrito Federal,378.0,433.0,419.0,449.0,467.0,436.0,581.0,644.0,741.0,706.0,2026-02-14 03:03:34.246875+00:00
1,Águas Claras,16.0,16.0,14.0,12.0,11.0,15.0,26.0,10.0,28.0,16.0,2026-02-14 03:03:34.246875+00:00
2,Arniqueira,0.0,0.0,0.0,0.0,0.0,3.0,9.0,12.0,9.0,5.0,2026-02-14 03:03:34.246875+00:00
3,Brasília,68.0,72.0,66.0,56.0,79.0,49.0,79.0,97.0,103.0,107.0,2026-02-14 03:03:34.246875+00:00
4,Brazlândia,5.0,6.0,4.0,10.0,5.0,6.0,8.0,9.0,12.0,17.0,2026-02-14 03:03:34.246875+00:00
5,Candangolândia,3.0,2.0,4.0,1.0,4.0,8.0,1.0,1.0,0.0,2.0,2026-02-14 03:03:34.246875+00:00
6,Ceilândia,51.0,62.0,55.0,65.0,62.0,46.0,69.0,69.0,88.0,79.0,2026-02-14 03:03:34.246875+00:00
7,Cruzeiro,4.0,1.0,4.0,2.0,1.0,3.0,5.0,5.0,9.0,10.0,2026-02-14 03:03:34.246875+00:00
8,Fercal,2.0,0.0,2.0,3.0,0.0,2.0,3.0,5.0,2.0,2.0,2026-02-14 03:03:34.246875+00:00
9,Gama,16.0,22.0,25.0,18.0,25.0,23.0,29.0,39.0,32.0,37.0,2026-02-14 03:03:34.246875+00:00


In [506]:
df_injuria.drop("inserido_em", axis=1, inplace=True)

In [507]:
df_injuria = df_injuria.melt(
    id_vars="regiao",  # coluna fixa        
    var_name="ano",  # nome da nova coluna com os valores das colunas originais
    value_name="injuria_racial"  # nome da nova coluna com os valores das colunas originais
)
df_injuria.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 340 entries, 0 to 339
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   regiao          340 non-null    object 
 1   ano             340 non-null    object 
 2   injuria_racial  340 non-null    float64
dtypes: float64(1), object(2)
memory usage: 8.1+ KB


In [508]:
df_injuria[["ano"]] = (
    df_injuria[["ano"]].apply(pd.to_numeric, errors="coerce").astype("Int64")
)

In [509]:
list(df_injuria['regiao'].unique())

['Distrito Federal',
 'Águas Claras',
 'Arniqueira',
 'Brasília',
 'Brazlândia',
 'Candangolândia',
 'Ceilândia',
 'Cruzeiro',
 'Fercal',
 'Gama',
 'Guará',
 'Itapoã',
 'Jardim Botânico',
 'Lago Norte',
 'Lago Sul',
 'Núcleo Bandeirante',
 'Paranoá',
 'Park Way',
 'Planaltina',
 'Recanto das Emas',
 'Riacho Fundo',
 'Riacho Fundo II',
 'Samambaia',
 'Santa Maria',
 'São Sebastião',
 'SCIA/Estrutural',
 'SIA',
 'Sobradinho',
 'Sobradinho II',
 'Sol Nascente/Pôr do Sol',
 'Sudoeste/Octogonal',
 'Taguatinga',
 'Varjão',
 'Vicente Pires']

In [510]:
df_injuria["regiao"] = df_injuria["regiao"].str.strip().str.upper()


In [511]:
df_injuria = df_injuria[df_injuria["regiao"] != "DISTRITO FEDERAL"]
df_injuria

,regiao,ano,injuria_racial
1,ÁGUAS CLARAS,2015,16.0
2,ARNIQUEIRA,2015,0.0
3,BRASÍLIA,2015,68.0
4,BRAZLÂNDIA,2015,5.0
5,CANDANGOLÂNDIA,2015,3.0
...,...,...,...
335,SOL NASCENTE/PÔR DO SOL,2024,9.0
336,SUDOESTE/OCTOGONAL,2024,13.0
337,TAGUATINGA,2024,55.0
338,VARJÃO,2024,0.0


In [512]:
df_injuria.info()

<class 'pandas.core.frame.DataFrame'>
Index: 330 entries, 1 to 339
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   regiao          330 non-null    object 
 1   ano             330 non-null    Int64  
 2   injuria_racial  330 non-null    float64
dtypes: Int64(1), float64(1), object(1)
memory usage: 10.6+ KB


In [513]:
df_injuria.describe()   

,ano,injuria_racial
count,330.0,330.000000
mean,2019.5,15.921212
std,2.876643,18.669087
min,2015.0,0.000000
25%,2017.0,4.000000
50%,2019.5,9.500000
75%,2022.0,18.000000
max,2024.0,107.000000


In [514]:
colunas_racismo = set(df_racismo["regiao"])
colunas_injuria = set(df_injuria["regiao"])

print("Só no df_racismo:")
print(colunas_racismo - colunas_injuria)

print("Só no df_injuria:")
print(colunas_injuria - colunas_racismo)

Só no df_racismo:
set()
Só no df_injuria:
set()


In [515]:
df_racismo_ = pd.merge(
    df_racismo,
    df_injuria,
    on=["ano", "regiao"],
    how="outer",
)
df_racismo_

,regiao,ano,racismo,injuria_racial
0,ARNIQUEIRA,2015,0,0.0
1,BRASÍLIA,2015,1,68.0
2,BRAZLÂNDIA,2015,0,5.0
3,CANDANGOLÂNDIA,2015,0,3.0
4,CEILÂNDIA,2015,1,51.0
...,...,...,...,...
325,SÃO SEBASTIÃO,2024,1,28.0
326,TAGUATINGA,2024,2,55.0
327,VARJÃO,2024,0,0.0
328,VICENTE PIRES,2024,0,12.0


In [516]:
tabelas

['crimes_contra_mulher',
 'desaparecidos_idade_sexo',
 'desaparecimento_localizados',
 'desaparecimento_regiao',
 'populacao_df',
 'feminicidio',
 'furto_em_veiculo',
 'homicidio',
 'violencia_idosos',
 'violencia_idosos_mensais',
 'violencia_idosos_ocorrencias',
 'violencia_idosos_sexo',
 'injuria_racial',
 'latrocinio',
 'lesao_corporal_morte',
 'populacao_regiao_administrativa',
 'racismo',
 'roubo_comercio',
 'roubo_transporte_coletivo',
 'roubo_veiculo',
 'roubo_pedestre']

In [517]:
desaparecidos_idade_sexo = carregar_tabela("desaparecidos_idade_sexo")
desaparecidos_idade_sexo

,ano,faixa_etaria,sexo,quantidade,inserido_em
0,2017,ATÉ 11 ANOS,MASCULINO,82,2026-02-14 03:03:33.789293+00:00
1,2017,DE 12 A 17 ANOS,MASCULINO,338,2026-02-14 03:03:33.789293+00:00
2,2017,DE 18 A 30 ANOS,MASCULINO,439,2026-02-14 03:03:33.789293+00:00
3,2017,DE 31 A 50 ANOS,MASCULINO,474,2026-02-14 03:03:33.789293+00:00
4,2017,MAIS DE 50 ANOS,MASCULINO,180,2026-02-14 03:03:33.789293+00:00
5,2017,NÃO INFORMADO,MASCULINO,25,2026-02-14 03:03:33.789293+00:00
6,2018,ATÉ 11 ANOS,MASCULINO,54,2026-02-14 03:03:33.789293+00:00
7,2018,DE 12 A 17 ANOS,MASCULINO,306,2026-02-14 03:03:33.789293+00:00
8,2018,DE 18 A 30 ANOS,MASCULINO,391,2026-02-14 03:03:33.789293+00:00
9,2018,DE 31 A 50 ANOS,MASCULINO,483,2026-02-14 03:03:33.789293+00:00


In [518]:
desaparecidos_idade_sexo.drop("inserido_em", axis=1, inplace=True)
desaparecidos_idade_sexo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   ano           60 non-null     int64 
 1   faixa_etaria  60 non-null     object
 2   sexo          60 non-null     object
 3   quantidade    60 non-null     int64 
dtypes: int64(2), object(2)
memory usage: 2.0+ KB


In [519]:
df_desaparecimento_regiao = carregar_tabela("desaparecimento_regiao")
df_desaparecimento_regiao.drop("inserido_em", axis=1, inplace=True)
df_desaparecimento_regiao

,regiao_administrativa,ocorrencias_2020,ocorrencias_2021,variacao_absoluta,participacao_percentual_2021
0,CEILANDIA,297,284,-13,13.75
1,PLANALTINA,162,145,-17,7.02
2,TAGUATINGA,148,174,26,8.43
3,BRASILIA,144,127,-17,6.15
4,SAMAMBAIA,119,154,35,7.46
5,RECANTO DAS EMAS,109,129,20,6.25
6,SANTA MARIA,96,60,-36,2.91
7,GAMA,93,94,1,4.55
8,SAO SEBASTIAO,81,87,6,4.21
9,GUARA,70,56,-14,2.71


In [520]:
df_desaparecimento_regiao.columns

Index(['regiao_administrativa', 'ocorrencias_2020', 'ocorrencias_2021',
       'variacao_absoluta', 'participacao_percentual_2021'],
      dtype='object')

In [521]:
df_desaparecimento_regiao.drop(
    ["variacao_absoluta", "participacao_percentual_2021"], axis=1, inplace=True
)
df_desaparecimento_regiao.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 3 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   regiao_administrativa  33 non-null     object
 1   ocorrencias_2020       33 non-null     int64 
 2   ocorrencias_2021       33 non-null     int64 
dtypes: int64(2), object(1)
memory usage: 924.0+ bytes


In [522]:
df_desaparecimento_localizados = carregar_tabela("desaparecimento_localizados")
df_desaparecimento_localizados.drop("inserido_em", axis=1, inplace=True)
df_desaparecimento_localizados

,ano,faixa_etaria,status,quantidade
0,2021,ATÉ 11 ANOS,AINDA DESAPARECIDOS,0
1,2021,DE 12 A 17 ANOS,AINDA DESAPARECIDOS,0
2,2021,DE 18 A 30 ANOS,AINDA DESAPARECIDOS,93
3,2021,DE 31 A 50 ANOS,AINDA DESAPARECIDOS,175
4,2021,MAIS DE 50 ANOS,AINDA DESAPARECIDOS,81
5,2021,NÃO INFORMADO,AINDA DESAPARECIDOS,1
6,2021,ATÉ 11 ANOS,LOCALIZADOS,58
7,2021,DE 12 A 17 ANOS,LOCALIZADOS,452
8,2021,DE 18 A 30 ANOS,LOCALIZADOS,478
9,2021,DE 31 A 50 ANOS,LOCALIZADOS,558


In [523]:
df_desaparecimento_localizados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   ano           12 non-null     int64 
 1   faixa_etaria  12 non-null     object
 2   status        12 non-null     object
 3   quantidade    12 non-null     int64 
dtypes: int64(2), object(2)
memory usage: 516.0+ bytes
